# Assignment 01: Data preparation and district analysis

**Overall purpose:** Answer the five crime-data questions on slide 11 using reproducible topic modeling and evidence-based report analysis. Your section must connect to the shared final report table and the main workstation.

## Shared rules: read before writing code

1. **Claim one assignment.** Add your name and reviewer below, then tell the group. These six notebooks are roles, not six tasks for each person.
2. **Use the shared files.** The original Parquet and shared/reports_base.parquet are read-only inputs. The base adds stable report_id values but does not clean or classify anything.
3. **Keep report_id.** Never recreate IDs after filtering, sorting or resetting an index. Join results using report_id with merge validation. URL is not a unique ID: some URLs repeat.
4. **Use one prepared version.** Member 1 must publish outputs/member_01/reports_prepared.parquet before final modeling. Preserve all source rows and mark exclusions. Everyone records that file's checksum in run_metadata.json. Prototype results on the base are exploratory and must be rerun on the agreed prepared file.
5. **Preserve alignment.** Build each text list and embedding matrix from the same filtered DataFrame in the same order. Never filter/sort text and embeddings separately.
6. **Separate analysis from evidence.** Write short code cells, show readable tables/plots, and explain actual outputs. Do not invent findings, silently fill missing districts, or treat a colored cluster plot as proof of meaningful crime types.
7. **Export to your own folder.** outputs/member_XX holds submissions; figures/member_XX holds readable figures. templates/member_XX contains empty examples and is never a finished submission.
8. **Finish with a peer review.** Everyone contributes their section, interpretation and sample reading. Member 6 coordinates validation; they do not write the entire group project.

**Your name:** Meriam El Askri  
**Peer reviewer:** Not assigned — pending  
**Status:** In progress — local preparation for review; final geographic analyses pending  
**Prepared input version/checksum:** Local review artifact; SHA-256 bf64238b7f929551cf20dadcdd314fa8bf17efe2cd7eab6174fd44dab1a1c139


## Setup and input check

Keep this notebook inside team_notebooks, or beside the shared folder. the notebook alone does not contain the dataset.

In [1]:
from pathlib import Path
import pandas as pd
import json

In [2]:
project_folder = Path.cwd()
if not (project_folder / "shared").exists():
    project_folder = project_folder.parent

assert (project_folder / "shared/source_manifest.json").exists(), "Open the notebook from the shared project folder."

In [3]:
output_folder = project_folder / "outputs/member_01"
figure_folder = project_folder / "figures/member_01"
output_folder.mkdir(parents=True, exist_ok=True)
figure_folder.mkdir(parents=True, exist_ok=True)

In [4]:
reports_base = pd.read_parquet(project_folder / "shared/reports_base.parquet")
assert reports_base["report_id"].is_unique
print("Base reports:", len(reports_base))
reports_base[["report_id", "date_posted", "title_en", "district"]].head(3)

Base reports: 5126


,report_id,date_posted,title_en,district
0,BR000000,2026-09-03,Police release wanted photo after shooting in ...,Woltmershausen
1,BR000001,2026-09-03,Burglary in Bar - Police Catch Perpetrators,Ostertor
2,BR000002,2026-09-02,Paint vandalism at high school,Horn


In [5]:
prepared_path = project_folder / "outputs/member_01/reports_prepared.parquet"
if prepared_path.exists():
    reports = pd.read_parquet(prepared_path)
    print("Prepared input available. Check its version before modeling.")
else:
    print("Preparation pending: final modeling and group analysis must wait for Member 1.")

Prepared input available. Check its version before modeling.


### Packages

Use the agreed group environment. From the project folder, the setup command is `python -m pip install -r requirements.txt`. Do this only when preparing the modeling environment. The skeleton uses pandas/pyarrow; BERTopic, UMAP and HDBSCAN are not installed in the environment used to check these starter notebooks. No algorithm has been run yet.

## Things to note throughout the project

- Count reports, not confirmed incidents or individual victims. One report can discuss several events or offenses. Treat the unit consistently.
- date_posted is a publication date. The first/last years and boundary months are partial; the source's collection coverage is not established by its date range alone.
- District labels include literal None and potentially different geographic levels. Unknown district reports still matter for citywide totals; district rankings need a consistent reviewed geography.
- Categories such as Theft, Traffic-related, Assault, Burglary, Fraud and Drug-related are candidate names, not predetermined model results. Include an Unknown/Unclear option and a documented non-crime/report-type category where needed.
- A topic is a group of similar reports, not automatically a legal crime category. Several topics can map to one category; mixed topics may need a per-report review.
- Use one primary category per report for the main count tables. If the group later adopts multiple labels, document that totals can exceed the report count and rebuild all comparisons consistently.
- The source includes descriptions of harmful events. Keep relevant evidence brief in outputs, avoid unnecessary identifying details, and use report IDs/source URLs for traceability.
- No API calls, paid services, extra scraping or AI-generated embeddings are needed to start. Optional AI-assisted elderly detection needs a separate agreed method and human verification.


## Purpose and dependencies

You prepare the shared dataset that everyone relies on, then return later to analyze district patterns. Errors here affect all four models and every later chart.

**You may start now:** use reports_base. **Your first handover:** prepared data, quality checks, district lookup and duplicate decisions. **Your later dependency:** wait for Member 6's approved report_categories before producing district/category results.

You answer the district-distribution part of Question 1 and the highest-district/composition question (Question 4). Your role has two phases; completing preparation does not complete the district analysis.

### 1. Audit the source

Check rows/columns, missing and blank text, invalid dates, district labels, repeated URLs and repeated title/description combinations. Check every embedding is a finite vector of length 768. Record issues rather than silently changing them. Keep both text fields and both embeddings.

In [6]:
with open(project_folder / "shared/source_manifest.json", encoding="utf-8") as file:
    source_manifest = json.load(file)

# The base has seven original source columns plus the permanent report_id.
expected_columns = ["report_id"] + source_manifest["original_columns"]
print("Dataset shape:", reports_base.shape)
print("Columns:", reports_base.columns.tolist())
print("Expected row count matches:", len(reports_base) == source_manifest["row_count"])
print("Expected column count matches:", len(reports_base.columns) == len(expected_columns))
print("Missing columns:", sorted(set(expected_columns) - set(reports_base.columns)))
print("Unexpected columns:", sorted(set(reports_base.columns) - set(expected_columns)))
print("Column order matches:", reports_base.columns.tolist() == expected_columns)

Dataset shape: (5126, 8)
Columns: ['report_id', 'date_posted', 'title_en', 'description_en', 'district', 'desc_en_embeddings768', 'title_en_embeddings768', 'url']
Expected row count matches: True
Expected column count matches: True
Missing columns: []
Unexpected columns: []
Column order matches: True


In [7]:
reports = reports_base.copy()

# Count missing values and blank or whitespace-only text separately.
for column in ["title_en", "description_en"]:
    missing_count = reports[column].isna().sum()
    blank_count = reports[column].str.strip().eq("").sum()
    print(f"{column}: {missing_count} missing, {blank_count} blank")

title_en: 0 missing, 0 blank
description_en: 0 missing, 0 blank


In [8]:
dates = reports_base["date_posted"]
missing_dates = dates.isna()
blank_dates = dates.str.strip().eq("").fillna(False)

# Convert a temporary copy for checking; keep the original dates unchanged.
parsed_dates = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
invalid_dates = parsed_dates.isna() & ~missing_dates & ~blank_dates

print("Valid dates:", parsed_dates.notna().sum())
print("Missing dates:", missing_dates.sum())
print("Blank dates:", blank_dates.sum())
print("Invalid dates:", invalid_dates.sum())
if invalid_dates.any():
    print(reports_base.loc[invalid_dates, ["report_id", "date_posted"]].head())

Valid dates: 5126
Missing dates: 0
Blank dates: 0
Invalid dates: 0


In [9]:
districts = reports_base["district"]
print("District column: district")
print("Missing districts:", districts.isna().sum())
print("Blank districts:", districts.str.strip().eq("").sum())
print("Unique district labels:", districts.nunique(dropna=False))
print(districts.value_counts(dropna=False).sort_index().to_string())

# Compare labels without changing the source values.
labels = districts.dropna().drop_duplicates().sort_values()
print("Labels with extra spaces:", labels[labels.ne(labels.str.strip()) | labels.str.contains(r"\s{2,}")].tolist())
label_check = pd.DataFrame({"original": labels, "comparison": labels.str.strip().str.replace(r"\s+", " ", regex=True).str.casefold()})
print("Possible spacing or capitalization variants:")
print(label_check[label_check["comparison"].duplicated(keep=False)].to_string(index=False))

District column: district
Missing districts: 0
Blank districts: 0
Unique district labels: 273
district
Achim                                             1
Achterdiek                                        1
Alte Neustadt                                   101
Alte Neustadt, Altstadt                           1
Alte Neustadt, Bahnhofsvorstadt                   1
Altstadt                                        310
Altstadt, Bahnhofsvorstadt                        1
Altstadt, Bahnhofsvorstadt, Neue Vahr Südost      1
Altstadt, Oslebshausen                            1
Altstadt/Bahnhofsvorstadt                         1
Anschlussstelle Horn                              1
Arbergen                                         17
Arsten                                           37
Arsten/ Kattenesch                                1
Aumund - Hammersbeck                              1
Aumund-Hammersbeck                               31
Autobahn 1                                        3
Autobahn 1 (A

In [10]:
# Compare exact values; ignore missing and blank entries.
for column in ["report_id", "url", "title_en", "description_en"]:
    values = reports_base[column]
    nonblank = values.notna() & values.str.strip().ne("")
    repeated = nonblank & values.duplicated(keep=False)
    extra_rows = values[nonblank].duplicated().sum()
    print(f"\n{column}: {values[repeated].nunique()} repeated values, "
          f"{repeated.sum()} reports involved, {extra_rows} repeats after the first")

    # Show two repeated groups with up to three reports each.
    for value in values[repeated].drop_duplicates().head(2):
        examples = reports_base.loc[values.eq(value), ["report_id", column]]
        if column == "report_id":
            examples = reports_base.loc[values.eq(value), ["report_id", "title_en"]]
        print(examples.head(3).to_string(index=False, max_colwidth=100))


report_id: 0 repeated values, 0 reports involved, 0 repeats after the first

url: 2 repeated values, 4 reports involved, 2 repeats after the first
report_id                                                    url
 BR001568 https://www.presseportal.de/blaulicht/pm/35235/5888850
 BR001569 https://www.presseportal.de/blaulicht/pm/35235/5888850
report_id                                                    url
 BR002113 https://www.presseportal.de/blaulicht/pm/35235/5714928
 BR002114 https://www.presseportal.de/blaulicht/pm/35235/5714928

title_en: 287 repeated values, 819 reports involved, 532 repeats after the first
report_id                     title_en
 BR000009 Man with gunshot wound found
 BR000175 Man with gunshot wound found
report_id             title_en
 BR000011 Injured with a knife
 BR000172 Injured with a knife
 BR000959 Injured with a knife

description_en: 2 repeated values, 4 reports involved, 2 repeats after the first
report_id                                                

In [11]:
# A repeated pair must match both text fields exactly.
text_columns = ["title_en", "description_en"]
repeated_pairs = reports_base.duplicated(subset=text_columns, keep=False)
pair_reports = reports_base.loc[repeated_pairs, ["report_id"] + text_columns]

print("Repeated title-description pairs:", pair_reports[text_columns].drop_duplicates().shape[0])
print("Reports involved:", len(pair_reports))
for pair, group in pair_reports.groupby(text_columns, sort=False):
    print("Matching report IDs:", ", ".join(group["report_id"]))

Repeated title-description pairs: 2
Reports involved: 4
Matching report IDs: BR001568, BR001569
Matching report IDs: BR002113, BR002114


In [12]:
import numpy as np
from numbers import Real

embedding_columns = [column for column in reports_base.columns if "embedding" in column]
print("Embedding columns:", embedding_columns)

Embedding columns: ['desc_en_embeddings768', 'title_en_embeddings768']


In [13]:
for column in embedding_columns:
    counts = {"missing": 0, "wrong_shape": 0, "non_numeric": 0,
              "contains_nan": 0, "contains_infinity": 0, "valid": 0}
    for vector in reports_base[column]:
        if pd.api.types.is_scalar(vector) and pd.isna(vector):
            counts["missing"] += 1
            continue

        # Inspect a temporary array without changing the stored vector.
        values = np.asarray(vector)
        wrong_shape = values.shape != (768,)
        numeric_values = [value for value in values.ravel() if isinstance(value, Real)]
        non_numeric = len(numeric_values) != values.size
        has_nan = bool(np.isnan(numeric_values).any())
        has_infinity = bool(np.isinf(numeric_values).any())
        counts["wrong_shape"] += wrong_shape
        counts["non_numeric"] += non_numeric
        counts["contains_nan"] += has_nan
        counts["contains_infinity"] += has_infinity
        counts["valid"] += not (wrong_shape or non_numeric or has_nan or has_infinity)

    print(f"\n{column} ? counts of reports")
    print(pd.Series(counts).to_string())


desc_en_embeddings768 ? counts of reports
missing                 0
wrong_shape             0
non_numeric             0
contains_nan            0
contains_infinity       0
valid                5126

title_en_embeddings768 ? counts of reports
missing                 0
wrong_shape             0
non_numeric             0
contains_nan            0
contains_infinity       0
valid                5126


**Explain the output:**

The dataset has 5,126 reports and 8 columns, matching the source manifest. There are no missing or blank titles or descriptions, all dates are valid, and every report ID is unique. Both embedding columns contain valid 768-dimensional vectors.

The 273 district labels include literal `None` and `null`, spelling variations, and combined locations. These labels need careful standardization before comparing districts. Two possible duplicate pairs have matching URLs, titles, and descriptions, but they need review before any exclusion decisions. No reports or source values have been changed.

### 2. Prepare dates and inclusion flags

Convert date_posted with pd.to_datetime and inspect any failures. Retain every base row/report_id. Add include_in_analysis (boolean) and exclusion_reason. Review duplicate URLs: repeated URL alone is not enough to delete a row. If excluding a confirmed duplicate, retain its row with a reason. Missing district alone should not exclude a report from text modeling or citywide analysis.

In [14]:
# Rebuild from source each time; keep every permanent ID.
reports = reports_base.copy()
reports["date_posted"] = pd.to_datetime(
    reports["date_posted"], format="%Y-%m-%d", errors="raise"
)
reports["include_in_analysis"] = True
reports["exclusion_reason"] = ""
duplicate_retained = {"BR001569": "BR001568", "BR002114": "BR002113"}
comparison_columns = ["date_posted", "title_en", "description_en", "url", "district"]
for excluded_id, retained_id in duplicate_retained.items():
    pair = reports_base.set_index("report_id").loc[[retained_id, excluded_id]]
    assert pair[comparison_columns].nunique(dropna=False).eq(1).all()
    mask = reports["report_id"].eq(excluded_id)
    reports.loc[mask, "include_in_analysis"] = False
    reports.loc[mask, "exclusion_reason"] = "Duplicate of retained matching report " + retained_id
assert len(reports) == 5126 and reports["date_posted"].notna().all()
print("Dates:", reports["date_posted"].notna().sum(), reports["date_posted"].dtype)
print("Included:", reports["include_in_analysis"].sum(), "| Excluded:", (~reports["include_in_analysis"]).sum())

Dates: 5126 datetime64[us]
Included: 5124 | Excluded: 2


In [15]:
# Compare only the two candidate pairs; do not change any records.
pairs = [("BR001568", "BR001569"), ("BR002113", "BR002114")]
review_columns = ["date_posted", "title_en", "description_en", "url", "district"]
for report_ids in pairs:
    pair_reports = reports_base.loc[reports_base["report_id"].isin(report_ids)]
    assert len(pair_reports) == 2
    print("\nPair:", ", ".join(report_ids))
    for column in review_columns:
        matches = pair_reports[column].nunique(dropna=False) == 1
        print(f"{column}: {'same' if matches else 'different'}")
        if column == "description_en":
            print("Description lengths:", pair_reports[column].str.len().tolist())
        else:
            print(pair_reports[["report_id", column]].to_string(index=False))


Pair: BR001568, BR001569
date_posted: same
report_id date_posted
 BR001568  2024-10-17
 BR001569  2024-10-17
title_en: same
report_id                 title_en
 BR001568 Robbery at Jewelry Store
 BR001569 Robbery at Jewelry Store
description_en: same
Description lengths: [984, 984]
url: same
report_id                                                    url
 BR001568 https://www.presseportal.de/blaulicht/pm/35235/5888850
 BR001569 https://www.presseportal.de/blaulicht/pm/35235/5888850
district: same
report_id district
 BR001568 Altstadt
 BR001569 Altstadt

Pair: BR002113, BR002114
date_posted: same
report_id date_posted
 BR002113  2024-02-15
 BR002114  2024-02-15
title_en: same
report_id           title_en
 BR002113 Supermarket robbed
 BR002114 Supermarket robbed
description_en: same
Description lengths: [767, 767]
url: same
report_id                                                    url
 BR002113 https://www.presseportal.de/blaulicht/pm/35235/5714928
 BR002114 https://www.presseportal.

**Explain your output:**

All 5,126 publication dates converted successfully to datetime. Both matching pairs remain in the dataset with their original IDs and content. The approved duplicate policy excludes BR001569 and BR002114 from analysis, retaining BR001568 and BR002113. Reasons name the retained matching report. There are 5,124 included reports and two excluded rows; no rows are deleted.

The shared geography is **Stadtteile plus standalone Ortsteile**. Both geographic analyses stay separate and use identical inclusion flags. Unresolved locations become Unknown, not exclusions. All four report-specific geographic exceptions remain protected. These methodological decisions are approved for implementation; peer review and publication of the local handover remain pending.

### 3. Review district names

Create district_clean using a documented lookup. Treat literal None/blank as Unknown. Agree one geography level: the source may mix neighborhoods and broader districts. Review spelling variants and places outside the agreed study area. Preserve district unchanged. Do not guess locations from the report title. Unknown remains visible in citywide totals.

#### Part 1: Ortsteile preliminary proposals

These are exact-name proposals only, not an applied lookup or an approved handover. Original labels are preserved exactly. Reference: [Bremen street directory, indexed November 2025 version](https://www.statistik.bremen.de/sixcms/media.php/13/Strassenverzeichnis_Bremen.pdf), cross-checked with [Die Ortsteile der Stadt Bremen 2015, published May 2016](https://www.statistik.bremen.de/sixcms/media.php/13/Ortsteilheft2015.pdf). The latest directory version still needs confirmation.

Names shared with Stadtteile and the Oberneuland special case are held back. Lehe, Bürgerpark, Radio Bremen and Regensburger Straße are also held back for city/place-context review. Spelling candidates, combined locations and unclear labels are not confirmed matches here. No report values or district_clean are changed.

In [16]:
# Exact official names only; ambiguous labels remain outside this review table.
ortsteil_names = """Alte Neustadt|Altstadt|Arbergen|Arsten|Aumund-Hammersbeck|Bahnhofsvorstadt|Barkhof|
Blockdiek|Blockland|Borgfeld|Buntentor|Burg-Grambke|Burgdamm|Ellener Feld|
Ellenerbrok-Schevemoor|Farge|Fesenfeld|Findorff-Bürgerweide|Fähr-Lobbendorf|Gartenstadt Süd|Gartenstadt Vahr|
Gete|Grohn|Grolland|Habenhausen|Hastedt|Hohentor|Hohweg|
Horn|Huckelriede|Hulsberg|In den Hufen|In den Wischen|Industriehäfen|Kattenesch|
Kattenturm|Kirchhuchting|Lehesterdeich|Lesum|Lindenhof|Lüssum-Bockhorn|Mahndorf|
Mittelshuchting|Neu-Schwachhausen|Neue Vahr Nord|Neue Vahr Südost|Neue Vahr Südwest|Neuenland|Neustädter Hafen|
Ohlenhof|Oslebshausen|Osterfeuerberg|Ostertor|Peterswerder|Rablinghausen|Rekum|
Riensberg|Rönnebeck|Schönebeck|Sebaldsbrück|Seehausen|Sodenmatt|St. Magnus|
Steffensweg|Steintor|Strom|Südervorstadt|Tenever|Utbremen|Weidedamm|
Werderland|Westend|Überseestadt""".replace("\n", "").split("|")

In [17]:
# This table proposes matches only; it does not update reports.
ortsteil_review = reports_base.loc[
    reports_base["district"].isin(ortsteil_names), "district"
].value_counts().rename_axis("district_original").reset_index(name="report_count")
ortsteil_review["proposed_ortsteil"] = ortsteil_review["district_original"]
ortsteil_review["reason"] = "Exact official Ortsteil name match"
ortsteil_review = ortsteil_review[
    ["district_original", "proposed_ortsteil", "reason", "report_count"]
].sort_values("district_original")

print("Proposed original labels:", len(ortsteil_review))
print("Reports covered:", ortsteil_review["report_count"].sum())
print(ortsteil_review.head(10).to_string(index=False))

Proposed original labels: 73
Reports covered: 2914
 district_original  proposed_ortsteil                             reason  report_count
     Alte Neustadt      Alte Neustadt Exact official Ortsteil name match           101
          Altstadt           Altstadt Exact official Ortsteil name match           310
          Arbergen           Arbergen Exact official Ortsteil name match            17
            Arsten             Arsten Exact official Ortsteil name match            37
Aumund-Hammersbeck Aumund-Hammersbeck Exact official Ortsteil name match            31
  Bahnhofsvorstadt   Bahnhofsvorstadt Exact official Ortsteil name match           387
           Barkhof            Barkhof Exact official Ortsteil name match            22
         Blockdiek          Blockdiek Exact official Ortsteil name match            28
         Blockland          Blockland Exact official Ortsteil name match             3
          Borgfeld           Borgfeld Exact official Ortsteil name match       

#### Review of four held-back labels

I reviewed all 69 reports' titles, descriptions, publication dates and linked Polizei Bremen releases. The original location headers name the Bremen Ortsteil for the single-location cases, resolving the city/park/broadcaster/street-name concerns. These are review recommendations, not applied mappings.

Keep BR001277 (Lehe and Barkhof), BR003479 (Lehe and Neue Vahr Südwest), and BR003518 (Bürgerpark and Gröpelingen) unresolved because they cover separate incidents in different Ortsteile. BR001797 describes an incident behind the Parkhotel and a later sighting at the Breminale/Osterdeich; keep it unresolved until the primary-location rule is agreed.

The other 65 reports have enough evidence to retain their original Ortsteil label, based on their descriptions and linked police location headers. BR002414 is a missing-person update: geographic support does not make it a confirmed crime report. This review does not change eligibility.

References: [official Bremen hierarchy](https://www.statistik.bremen.de/sixcms/media.php/13/Strassenverzeichnis_Bremen.pdf); [BR000264](https://www.presseportal.de/blaulicht/pm/35235/6269808); [BR000859](https://www.presseportal.de/blaulicht/pm/35235/6103497); [BR001277](https://www.presseportal.de/blaulicht/pm/35235/5982320); [BR003479](https://www.presseportal.de/blaulicht/pm/35235/5210085); [BR003518](https://www.presseportal.de/blaulicht/pm/35235/5196378); [BR001797](https://www.presseportal.de/blaulicht/pm/35235/5819520). The dataset's original URLs remain the source references for every reviewed report.

In [18]:
# Record the manual review only; do not apply any geographic mapping.
held_labels = ["Lehe", "Bürgerpark", "Radio Bremen", "Regensburger Straße"]
unresolved_ids = ["BR001277", "BR003479", "BR003518", "BR001797"]
held_review = reports_base.loc[
    reports_base["district"].isin(held_labels),
    ["report_id", "district", "date_posted", "title_en", "url"]
].copy()
held_review["review_status"] = "supported"
held_review.loc[held_review["report_id"].isin(unresolved_ids), "review_status"] = "needs_review"
assert len(held_review) == 69
print(pd.crosstab(held_review["district"], held_review["review_status"]).to_string())

example_ids = ["BR000068", "BR000859", "BR000264", "BR001277", "BR003518", "BR001797"]
print(held_review.loc[held_review["report_id"].isin(example_ids),
                      ["report_id", "district", "date_posted", "review_status"]].to_string(index=False))

review_status        needs_review  supported
district                                    
Bürgerpark                      2          7
Lehe                            2         45
Radio Bremen                    0         11
Regensburger Straße             0          2
report_id            district date_posted review_status
 BR000068                Lehe  2026-08-07     supported
 BR000264 Regensburger Straße  2026-05-06     supported
 BR000859        Radio Bremen  2025-08-25     supported
 BR001277                Lehe  2025-03-03  needs_review
 BR001797          Bürgerpark  2024-07-09  needs_review
 BR003518          Bürgerpark  2022-04-13  needs_review


#### Part 1: complete Ortsteil label review preliminary, pending review

The table `ortsteil_label_review` covers every original district label, including unresolved cases. It extends the 73 exact-name proposals and reuses the earlier four-label review without changing either result. It is a review table, not the contract's final district lookup, and no mapping is applied.

Each row records the original label, report count, category, proposed Ortsteil or unresolved outcome, candidate spelling where relevant, decision, reason and evidence. Original labels, including embedded control characters, remain unchanged. Candidate spellings are never treated as accepted locations. Exact-name proposals verify the name against the reference; they do not claim every underlying report has been manually checked.

References: [official Bremen street directory, indexed November 2025 version](https://www.statistik.bremen.de/sixcms/media.php/13/Strassenverzeichnis_Bremen.pdf), cross-checked with [Die Ortsteile der Stadt Bremen 2015, published May 2016](https://www.statistik.bremen.de/sixcms/media.php/13/Ortsteilheft2015.pdf). The latest directory version and final boundary compatibility remain pending. For unresolved categories these references provide the vocabulary, not proof of a location. Source report IDs and URLs provide traceability; the earlier manual-review note explains the 69 report-context decisions.

The mixed-evidence category counts each original label and all its reports once. Its supported and unresolved counts are shown separately. The four exceptions are retained by report_id in `ortsteil_report_exceptions`. A proposed Ortsteil on a mixed row applies only to the supported reports, never to the exceptions. Supporting notes and original URLs remain available in the earlier `held_review` table.

Unknown information and uncertainty about location do not exclude a report. Other-municipality labels are review candidates, not automatic exclusions; harbour cases require special care. Standalone Ortsteile stay at Ortsteil level; no parent Stadtteile are invented here.

In [19]:
# Suggestions only: candidate names are not accepted Ortsteil mappings.
spelling_candidates = {
    "Aumund - Hammersbeck": "Aumund-Hammersbeck",
    "Bahnhofvorstadt": "Bahnhofsvorstadt",
    "Ellener-Feld": "Ellener Feld",
    "Ellenerbrok-Schevermoor": "Ellenerbrok-Schevemoor",
    "Ellernerbrok-Schevemoor": "Ellenerbrok-Schevemoor",
    "Ellernerbrook-Schevemoor": "Ellenerbrok-Schevemoor",
    "Findorf": "Findorff",
    "Gartenstadt-Süd": "Gartenstadt Süd",
    "Gröpelinger": "Gröpelingen",
    "Huckeriede": "Huckelriede",
    "Kirchhhuchting": "Kirchhuchting",
    "Lehersterdeich": "Lehesterdeich",
    "Neu Schwachausen": "Neu-Schwachhausen",
    "Neu Schwachhausen": "Neu-Schwachhausen",
    "Neu Vahr Nord": "Neue Vahr Nord",
    "Neue Vahr-Südwest": "Neue Vahr Südwest",
    "Oslebhausen": "Oslebshausen",
    "Petersweder": "Peterswerder",
    "Sodematt": "Sodenmatt",
    "B\nrgerpark": "Bürgerpark",
    "F\r-Lobbendorf": "Fähr-Lobbendorf"
}


In [20]:
shared_names = "Blumenthal|Gröpelingen|Hemelingen|Neustadt|Osterholz|Schwachhausen|Vegesack|Walle|Woltmershausen".split("|")
broader_names = "Burglesum|Findorff|Horn-Lehe|Huchting|Häfen|Mitte|Oberneuland|Obervieland|Vahr|Östliche Vorstadt".split("|")
other_places = """Achim|Berlin|Blexen Reede|Blexen-Reede|Bremerhaven|Brinkum|Fischereihafen|
Frankreich|Geestemünde-Nord|Hamburg-Neustadt|Ihlpohl|Ingolstadt|Italien|Sottrum|Varrel|Wremen""".replace("\n", "").split("|")
local_places = """Achterdiek|Anschlussstelle Horn|Autobahn 1|Autobahn 1 (A1)|Autobahn 27|Autobahn 270|
Bahnhofsplatz|Bundesautobahn 1|Bundesautobahn 27|Bundesautobahn A1|Bürgerweide|
Container Terminal Bremerhaven|Flughafen|Freimarkt|Hasenbüren|Hauptbahnhof|Kaiserhafen III|
Marktplatz|Neuer Hafen|Nordschleuse|Osthafen|Schlachte|Seehafen|Stromkaje|Verbindungshafen|
Weser|Weserstadion|Wohninvest Weserstadion|Wohninvest-Weserstadion|wohninvest Weserstadion|Überseehafen""".replace("\n", "").split("|")


In [21]:
district_labels = reports_base["district"].drop_duplicates()
combined_hyphens = """Findorff-Schwachhausen|Findorff-Weidedamm|Mitte-West|Mitte-Östliche Vorstadt|
Neustadt-Obervieland|Obervieland-Industriehäfen|Walle-Gröpelingen""".replace("\n", "").split("|")
combined_labels = district_labels[
    district_labels.str.contains(r",|/|\bund\b", regex=True) |
    district_labels.isin(combined_hyphens)
].tolist()

review_groups = {
    "Exact-name proposal": ortsteil_names,
    "Report-context supported": ["Radio Bremen", "Regensburger Straße"],
    "Mixed report-level evidence": ["Lehe", "Bürgerpark"],
    "Spelling candidate": list(spelling_candidates),
    "Ambiguous Ortsteil/Stadtteil name": shared_names,
    "Broader Stadtteil": broader_names,
    "Combined locations": combined_labels,
    "Landmark/local place": local_places,
    "Other municipality candidate": other_places,
    "Unknown literal": ["None", "null"]
}


In [22]:
# These decisions describe a review; they do not change the dataset.
review_reasons = {
    "Exact-name proposal": "Exact official Ortsteil name; label-name evidence only, not a fresh review of every report.",
    "Report-context supported": "All reports for this label were supported by the earlier report-specific police-source review.",
    "Mixed report-level evidence": "Use the earlier report-specific review; some reports need a primary-location decision.",
    "Spelling candidate": "Possible spelling, separator or control-character variation; candidate still needs confirmation.",
    "Ambiguous Ortsteil/Stadtteil name": "Official name at both levels; the label alone does not identify the smaller area.",
    "Broader Stadtteil": "A larger district does not establish one Ortsteil; keep unresolved without more evidence.",
    "Combined locations": "Multiple or hierarchical place names; do not split counts or choose a location automatically.",
    "Landmark/local place": "Place, road or landmark name needs precise location evidence and boundary verification.",
    "Other municipality candidate": "Appears to name another municipality or area; verify city scope, especially harbour cases.",
    "Unknown literal": "Literal missing-information label; retain the report and preserve its original label.",
    "Unclear/informal": "No verified unambiguous Ortsteil match established; further evidence is needed."
}
review_decisions = {
    "Exact-name proposal": "proposed_exact_match",
    "Report-context supported": "proposed_report_match",
    "Mixed report-level evidence": "report_specific_only",
    "Spelling candidate": "verify_candidate"
}


In [23]:
# One row for every original label; keep source spellings and control characters.
ortsteil_label_review = reports_base.groupby("district", dropna=False).agg(
    report_count=("report_id", "size"),
    sample_report_id=("report_id", "first"),
    sample_url=("url", "first")
).reset_index().rename(columns={"district": "district_original"})
ortsteil_label_review["review_category"] = "Unclear/informal"

grouped_labels = [label for labels in review_groups.values() for label in labels]
assert len(grouped_labels) == len(set(grouped_labels)), "Review groups overlap."
for category, labels in review_groups.items():
    mask = ortsteil_label_review["district_original"].isin(labels)
    ortsteil_label_review.loc[mask, "review_category"] = category

ortsteil_label_review["decision"] = ortsteil_label_review["review_category"].map(review_decisions).fillna("unresolved")
ortsteil_label_review["reason"] = ortsteil_label_review["review_category"].map(review_reasons)
ortsteil_label_review["candidate_name"] = ortsteil_label_review["district_original"].map(spelling_candidates).fillna("")
ortsteil_label_review["review_status"] = "preliminary_pending_review"


In [24]:
official_directory = "https://www.statistik.bremen.de/sixcms/media.php/13/Strassenverzeichnis_Bremen.pdf"
official_crosscheck = "https://www.statistik.bremen.de/sixcms/media.php/13/Ortsteilheft2015.pdf"
ortsteil_label_review["official_reference"] = official_directory + "; " + official_crosscheck
ortsteil_label_review["evidence"] = (
    "Original dataset label; sample " + ortsteil_label_review["sample_report_id"] +
    "; " + ortsteil_label_review["sample_url"]
)

# These cases need more specific notes than a general category reason.
special_notes = {
    "Findorf": "Candidate Findorff is a Stadtteil, not a confirmed Ortsteil; do not apply this candidate.",
    "Gröpelinger": "Candidate Gröpelingen names both levels; spelling and geographic meaning remain unverified.",
    "Oberneuland": "Special single-area/statistical treatment needs agreement; do not assume an Ortsteil mapping.",
    "Industriehafen": "Possible relation to Industriehäfen needs verification; no correction accepted.",
    "Neustädter Häfen": "Possible relation to Neustädter Hafen needs verification; no correction accepted.",
    "Überseehafengebiet": "Check the exact harbour territory and city boundary; do not infer a location."
}
for label, reason in special_notes.items():
    ortsteil_label_review.loc[ortsteil_label_review["district_original"].eq(label), "reason"] = reason


In [25]:
ortsteil_label_review["proposed_ortsteil"] = "Unresolved"
ortsteil_label_review["supported_reports"] = 0
exact_mask = ortsteil_label_review["review_category"].eq("Exact-name proposal")
ortsteil_label_review.loc[exact_mask, "proposed_ortsteil"] = ortsteil_label_review.loc[exact_mask, "district_original"]
ortsteil_label_review.loc[exact_mask, "supported_reports"] = ortsteil_label_review.loc[exact_mask, "report_count"]
unknown_mask = ortsteil_label_review["review_category"].eq("Unknown literal")
ortsteil_label_review.loc[unknown_mask, "proposed_ortsteil"] = "Unknown"
ortsteil_label_review["exception_report_ids"] = ""

# Reuse the 69-report manual review, including its four unresolved exceptions.
assert len(held_review) == 69 and held_review["report_id"].is_unique
for label, group in held_review.groupby("district"):
    mask = ortsteil_label_review["district_original"].eq(label)
    supported_count = group["review_status"].eq("supported").sum()
    exception_ids = group.loc[group["review_status"].eq("needs_review"), "report_id"]
    ortsteil_label_review.loc[mask, "supported_reports"] = supported_count
    ortsteil_label_review.loc[mask, "proposed_ortsteil"] = label
    ortsteil_label_review.loc[mask, "exception_report_ids"] = "; ".join(exception_ids)
    ortsteil_label_review.loc[mask, "evidence"] = "Earlier manual review of these report IDs: " + "; ".join(group["report_id"])
ortsteil_label_review["unresolved_reports"] = ortsteil_label_review["report_count"] - ortsteil_label_review["supported_reports"]


In [26]:
# Keep report-level exceptions separate from the label proposals.
exception_reasons = {
    "BR001277": "Separate incidents in Lehe and Barkhof; primary location unresolved.",
    "BR003479": "Separate incidents in Lehe and Neue Vahr Südwest; primary location unresolved.",
    "BR003518": "Separate incidents in Bürgerpark and Gröpelingen; primary location unresolved.",
    "BR001797": "Incident behind Parkhotel and later sighting at Osterdeich; primary-location rule pending."
}
ortsteil_report_exceptions = held_review.loc[
    held_review["review_status"].eq("needs_review")
].copy()
ortsteil_report_exceptions["proposed_ortsteil"] = "Unresolved"
ortsteil_report_exceptions["decision"] = "needs_review"
ortsteil_report_exceptions["reason"] = ortsteil_report_exceptions["report_id"].map(exception_reasons)
ortsteil_report_exceptions["evidence"] = ortsteil_report_exceptions["url"]


In [27]:
assert len(ortsteil_label_review) == 273
assert ortsteil_label_review["district_original"].is_unique
assert set(ortsteil_label_review["district_original"]) == set(reports_base["district"])
assert ortsteil_label_review["report_count"].sum() == len(reports_base) == 5126
assert len(ortsteil_names) == 73 and exact_mask.sum() == 73
assert ortsteil_label_review.loc[exact_mask, "report_count"].sum() == 2914
assert held_review["review_status"].eq("supported").sum() == 65
assert set(ortsteil_report_exceptions["report_id"]) == set(exception_reasons)
assert ortsteil_label_review["reason"].notna().all()
assert ortsteil_label_review["evidence"].str.len().gt(0).all()
assert (ortsteil_label_review["supported_reports"] + ortsteil_label_review["unresolved_reports"]).equals(ortsteil_label_review["report_count"])

ortsteil_review_summary = ortsteil_label_review.groupby("review_category").agg(
    original_labels=("district_original", "size"), reports=("report_count", "sum"),
    supported_proposals=("supported_reports", "sum"), unresolved_reports=("unresolved_reports", "sum")
)
print(ortsteil_review_summary.to_string())
print("\nTotal labels:", len(ortsteil_label_review), "| Total reports:", ortsteil_label_review["report_count"].sum())
print("Supported preliminary proposals:", ortsteil_label_review["supported_reports"].sum())


                                   original_labels  reports  supported_proposals  unresolved_reports
review_category                                                                                     
Ambiguous Ortsteil/Stadtteil name                9      883                    0                 883
Broader Stadtteil                               10      408                    0                 408
Combined locations                              88       99                    0                  99
Exact-name proposal                             73     2914                 2914                   0
Landmark/local place                            31       68                    0                  68
Mixed report-level evidence                      2       56                   52                   4
Other municipality candidate                    16       31                    0                  31
Report-context supported                         2       13                   13           

In [28]:
preview_columns = ["district_original", "report_count", "proposed_ortsteil", "decision"]
print("Preliminary label review preview:")
print(ortsteil_label_review[preview_columns].head(8).to_string(index=False))
print("\nReport-specific exceptions (never apply a blanket label mapping):")
print(ortsteil_report_exceptions[["report_id", "district", "reason"]].to_string(index=False))


Preliminary label review preview:
                           district_original  report_count proposed_ortsteil             decision
                                       Achim             1        Unresolved           unresolved
                                  Achterdiek             1        Unresolved           unresolved
                               Alte Neustadt           101     Alte Neustadt proposed_exact_match
                     Alte Neustadt, Altstadt             1        Unresolved           unresolved
             Alte Neustadt, Bahnhofsvorstadt             1        Unresolved           unresolved
                                    Altstadt           310          Altstadt proposed_exact_match
                  Altstadt, Bahnhofsvorstadt             1        Unresolved           unresolved
Altstadt, Bahnhofsvorstadt, Neue Vahr Südost             1        Unresolved           unresolved

Report-specific exceptions (never apply a blanket label mapping):
report_id   distr

#### Part 2: complete Stadtteil label review preliminary, pending review

`stadtteil_label_review` is separate from Part 1 and covers all original labels. It proposes a larger district only when the exact label or earlier report-context review supports it. The official hierarchy establishes the parent relationship; a spelling candidate does not establish that the report belongs there.

Sources: the Statistisches Landesamt Bremen's [street directory, indexed PDF version November 2025](https://www.statistik.bremen.de/sixcms/media.php/13/Strassenverzeichnis_Bremen.pdf), cross-checked against [Die Ortsteile der Stadt Bremen 2015, published May 2016](https://www.statistik.bremen.de/sixcms/media.php/13/Ortsteilheft2015.pdf). Source/version fields and report evidence are stored in the review table. Codes below are the two-digit Stadtteil and three-digit Ortsteil identifiers used by these references, scoped to the city of Bremen. Reference codes identify a matching official name, not proof of the report's finer geographic precision. The latest reference version and compatibility with eventual map boundaries remain unverified. Canonical Kattenesch (234) and Lehesterdeich (343) spellings were cross-checked because the indexed street-directory overview contains apparent spelling errors.

Same-name Ortsteil/Stadtteil cases can support the larger district without confirming the smaller area, since both Bremen interpretations lead to the same Stadtteil. Mitte is different: Stadtteil Mitte (11) and Stadtbezirk Mitte (1) have different extents. Its reports stay unresolved pending context review. Oberneuland is treated separately as official Stadtteil 36; no finer relationship is invented.

Borgfeld (351), Blockland (411), Seehausen (261) and Strom (271) are standalone Ortsteile. They retain their official identities but have no proposed Stadtteil or invented parent code. They are counted separately among reports without a Stadtteil proposal; this is a structural exception, not missing location information.

All four Part 1 report-specific exceptions remain unresolved here because the known locations span different Stadtteile. The separate `stadtteil_report_exceptions` table records those reasons and original URLs. A label's proposed Stadtteil never overrides these cases. Combined/hierarchical labels, spelling candidates and landmarks remain unresolved until evidence is reviewed; candidate parent names/codes are clearly separate from accepted preliminary proposals. Other-municipality candidates are not exclusions, especially around harbour territories.

A detached `stadtteil_report_review` validates one proposal outcome per original report_id. It does not update reports, district_clean, inclusion flags or any shared dataset. All original labels and reports are retained. No totals from Part 1 and Part 2 are added together. No tables are exported and no contract schemas are changed.

In [29]:
# Fixed official references; no online downloads are needed when running the notebook.
stadtteil_reference_sources = pd.DataFrame([
    ["Bremen street directory", "Indexed PDF: November 2025", official_directory],
    ["Die Ortsteile der Stadt Bremen 2015", "Published May 2016", official_crosscheck]
], columns=["source", "version_or_publication", "url"])
stadtteil_codes = {
    "Mitte": "11",
    "Häfen": "12",
    "Neustadt": "21",
    "Obervieland": "23",
    "Huchting": "24",
    "Woltmershausen": "25",
    "Östliche Vorstadt": "31",
    "Schwachhausen": "32",
    "Vahr": "33",
    "Horn-Lehe": "34",
    "Oberneuland": "36",
    "Osterholz": "37",
    "Hemelingen": "38",
    "Findorff": "42",
    "Walle": "43",
    "Gröpelingen": "44",
    "Burglesum": "51",
    "Vegesack": "52",
    "Blumenthal": "53"
}


In [30]:
# Official Ortsteil codes and their named parent Stadtteile (city Bremen).
stadtteil_parent_groups = {
    "Mitte": "111:Altstadt|112:Bahnhofsvorstadt|113:Ostertor",
    "Häfen": "122:Industriehäfen|123:Stadtbremisches Überseehafengebiet Bremerhaven|124:Neustädter Hafen|125:Hohentorshafen",
    "Neustadt": "211:Alte Neustadt|212:Hohentor|213:Neustadt|214:Südervorstadt|215:Gartenstadt Süd|216:Buntentor|217:Neuenland|218:Huckelriede",
    "Obervieland": "231:Habenhausen|232:Arsten|233:Kattenturm|234:Kattenesch",
    "Huchting": "241:Mittelshuchting|242:Sodenmatt|243:Kirchhuchting|244:Grolland",
    "Woltmershausen": "251:Woltmershausen|252:Rablinghausen",
    "Östliche Vorstadt": "311:Steintor|312:Fesenfeld|313:Peterswerder|314:Hulsberg",
    "Schwachhausen": "321:Neu-Schwachhausen|322:Bürgerpark|323:Barkhof|324:Riensberg|325:Radio Bremen|326:Schwachhausen|327:Gete",
    "Vahr": "331:Gartenstadt Vahr|332:Neue Vahr Nord|334:Neue Vahr Südwest|335:Neue Vahr Südost"
}


In [31]:
stadtteil_parent_groups.update({
    "Horn-Lehe": "341:Horn|342:Lehe|343:Lehesterdeich",
    "Osterholz": "371:Ellener Feld|372:Ellenerbrok-Schevemoor|373:Tenever|374:Osterholz|375:Blockdiek",
    "Hemelingen": "381:Sebaldsbrück|382:Hastedt|383:Hemelingen|384:Arbergen|385:Mahndorf",
    "Findorff": "421:Regensburger Straße|422:Findorff-Bürgerweide|423:Weidedamm|424:In den Hufen",
    "Walle": "431:Utbremen|432:Steffensweg|433:Westend|434:Walle|435:Osterfeuerberg|436:Hohweg|437:Überseestadt",
    "Gröpelingen": "441:Lindenhof|442:Gröpelingen|443:Ohlenhof|444:In den Wischen|445:Oslebshausen",
    "Burglesum": "511:Burg-Grambke|512:Werderland|513:Burgdamm|514:Lesum|515:St. Magnus",
    "Vegesack": "521:Vegesack|522:Grohn|523:Schönebeck|524:Aumund-Hammersbeck|525:Fähr-Lobbendorf",
    "Blumenthal": "531:Blumenthal|532:Rönnebeck|533:Lüssum-Bockhorn|534:Farge|535:Rekum"
})
standalone_ortsteile = {"Borgfeld": "351", "Blockland": "411", "Seehausen": "261", "Strom": "271"}


In [32]:
stadtteil_hierarchy_rows = []
for parent_name, entries in stadtteil_parent_groups.items():
    for entry in entries.split("|"):
        ortsteil_code, ortsteil_name = entry.split(":", 1)
        stadtteil_hierarchy_rows.append([
            ortsteil_code, ortsteil_name, stadtteil_codes[parent_name], parent_name
        ])
for ortsteil_name, ortsteil_code in standalone_ortsteile.items():
    stadtteil_hierarchy_rows.append([ortsteil_code, ortsteil_name, "", ""])

stadtteil_official_hierarchy = pd.DataFrame(
    stadtteil_hierarchy_rows,
    columns=["ortsteil_code", "ortsteil_name", "stadtteil_code", "stadtteil_name"]
)
assert stadtteil_official_hierarchy["ortsteil_name"].is_unique
assert stadtteil_official_hierarchy["ortsteil_code"].is_unique
stadtteil_reference_by_name = stadtteil_official_hierarchy.set_index("ortsteil_name")


In [33]:
# Copy review information only; leave Part 1 and the working dataset unchanged.
stadtteil_label_review = ortsteil_label_review[
    ["district_original", "report_count", "sample_report_id", "sample_url",
     "candidate_name", "evidence", "exception_report_ids"]
].copy()
stadtteil_category_names = {
    "Exact-name proposal": "Ortsteil to official parent",
    "Report-context supported": "Report-context parent",
    "Mixed report-level evidence": "Mixed report-level evidence",
    "Ambiguous Ortsteil/Stadtteil name": "Same-name consistent Stadtteil",
    "Broader Stadtteil": "Direct Stadtteil name"
}
stadtteil_label_review["review_category"] = ortsteil_label_review["review_category"].replace(stadtteil_category_names)
stadtteil_special_categories = {
    "Standalone Ortsteil": list(standalone_ortsteile),
    "Oberneuland special case": ["Oberneuland"],
    "Ambiguous Mitte level": ["Mitte"],
    "Broad location": ["Bremen", "Bremen Nord", "Nord", "Ost", "Innenstadt", "Stadtgebiet"]
}
for category, names in stadtteil_special_categories.items():
    stadtteil_label_review.loc[stadtteil_label_review["district_original"].isin(names), "review_category"] = category


In [34]:
stadtteil_reasons = {
    "Ortsteil to official parent": "Exact-name Ortsteil proposal has this verified official parent; not a fresh manual review of every report.",
    "Report-context parent": "Earlier report-specific evidence supports the Ortsteil and its verified official parent.",
    "Mixed report-level evidence": "Parent proposed only for the supported reports; the four recorded exceptions stay unresolved.",
    "Same-name consistent Stadtteil": "Both Bremen name interpretations lead to the same Stadtteil; the smaller Ortsteil is not confirmed.",
    "Direct Stadtteil name": "Exact official Stadtteil name; a preliminary name-based proposal.",
    "Oberneuland special case": "Official Stadtteil Oberneuland, code 36; do not invent a finer geographic relationship.",
    "Ambiguous Mitte level": "Mitte names Stadtteil 11 and Stadtbezirk 1, which also includes Häfen; report context is needed.",
    "Standalone Ortsteil": "Official standalone Ortsteil without a parent Stadtteil; retain separately without inventing a parent.",
    "Spelling candidate": "Candidate spelling and any parent relationship remain unconfirmed; no proposal is accepted here.",
    "Combined locations": "Multiple or hierarchical locations need individual verification; do not split counts or infer a common parent.",
    "Landmark/local place": "Precise incident location and geographic boundary evidence are still needed.",
    "Other municipality candidate": "City scope remains to be verified; harbour cases may involve Bremen's enclave.",
    "Unknown literal": "Literal missing-information label; keep the report and its original district value.",
    "Broad location": "City, regional or informal location does not identify one Stadtteil.",
    "Unclear/informal": "No verified single Stadtteil established; further evidence is needed."
}
stadtteil_label_review["reason"] = stadtteil_label_review["review_category"].map(stadtteil_reasons)
stadtteil_label_review["decision"] = "unresolved"
stadtteil_label_review["review_status"] = "preliminary_pending_review"


In [35]:
stadtteil_labels = stadtteil_label_review["district_original"]
stadtteil_label_review["reference_ortsteil_code"] = stadtteil_labels.map(stadtteil_reference_by_name["ortsteil_code"]).fillna("")
stadtteil_label_review["reference_stadtteil_code"] = stadtteil_labels.map(stadtteil_codes).fillna("")
stadtteil_label_review["proposed_stadtteil"] = "Unresolved"
stadtteil_label_review["proposed_stadtteil_code"] = ""

stadtteil_parent_mask = stadtteil_label_review["review_category"].isin(
    ["Ortsteil to official parent", "Report-context parent", "Mixed report-level evidence"]
)
stadtteil_label_review.loc[stadtteil_parent_mask, "proposed_stadtteil"] = stadtteil_labels[stadtteil_parent_mask].map(stadtteil_reference_by_name["stadtteil_name"])
stadtteil_name_mask = stadtteil_label_review["review_category"].isin(
    ["Direct Stadtteil name", "Same-name consistent Stadtteil", "Oberneuland special case"]
)
stadtteil_label_review.loc[stadtteil_name_mask, "proposed_stadtteil"] = stadtteil_labels[stadtteil_name_mask]
stadtteil_label_review["proposed_stadtteil_code"] = stadtteil_label_review["proposed_stadtteil"].map(stadtteil_codes).fillna("")
stadtteil_supported_mask = stadtteil_label_review["proposed_stadtteil_code"].isin(stadtteil_codes.values())
stadtteil_label_review.loc[stadtteil_supported_mask, "decision"] = "preliminary_proposal"
stadtteil_label_review.loc[stadtteil_label_review["review_category"].eq("Mixed report-level evidence"), "decision"] = "report_specific_only"
stadtteil_label_review.loc[stadtteil_label_review["review_category"].eq("Standalone Ortsteil"), "proposed_stadtteil"] = "No parent Stadtteil"
stadtteil_label_review.loc[stadtteil_label_review["review_category"].eq("Unknown literal"), "proposed_stadtteil"] = "Unknown"


In [36]:
stadtteil_label_review["official_reference"] = official_directory + "; " + official_crosscheck
stadtteil_label_review["reference_version"] = "Street directory indexed 2025-11; 2015 edition published 2016-05"
stadtteil_label_review["reference_city"] = "Stadt Bremen"
stadtteil_label_review["reference_stadtbezirk_code"] = ""
stadtteil_label_review.loc[stadtteil_labels.eq("Mitte"), "reference_stadtbezirk_code"] = "1"
stadtteil_label_review["evidence"] = (
    stadtteil_label_review["evidence"] + "; sample source: " + stadtteil_label_review["sample_url"]
)

# Candidate parents are reference suggestions only, never accepted proposals.
stadtteil_candidate_parents = stadtteil_reference_by_name["stadtteil_name"].to_dict()
stadtteil_candidate_parents.update({name: name for name in stadtteil_codes})
stadtteil_label_review["candidate_stadtteil"] = stadtteil_label_review["candidate_name"].map(stadtteil_candidate_parents).fillna("")
stadtteil_label_review["candidate_stadtteil_code"] = stadtteil_label_review["candidate_stadtteil"].map(stadtteil_codes).fillna("")


In [37]:
# Preserve all four exceptions; their known locations span different Stadtteile.
stadtteil_report_exceptions = ortsteil_report_exceptions[
    ["report_id", "district", "url"]
].copy()
stadtteil_exception_reasons = {
    "BR001277": "Lehe/Horn-Lehe and Barkhof/Schwachhausen: no agreed primary Stadtteil.",
    "BR003479": "Lehe/Horn-Lehe and Neue Vahr Südwest/Vahr: no agreed primary Stadtteil.",
    "BR003518": "Bürgerpark/Schwachhausen and Gröpelingen: no agreed primary Stadtteil.",
    "BR001797": "Incident near Parkhotel/Schwachhausen and later sighting at Osterdeich/Mitte: primary-location rule pending."
}
stadtteil_report_exceptions["proposed_stadtteil"] = "Unresolved"
stadtteil_report_exceptions["proposed_stadtteil_code"] = ""
stadtteil_report_exceptions["decision"] = "needs_review"
stadtteil_report_exceptions["reason"] = stadtteil_report_exceptions["report_id"].map(stadtteil_exception_reasons)
stadtteil_report_exceptions["evidence"] = stadtteil_report_exceptions["url"]


In [38]:
# A detached proposal view validates report counts; reports and district_clean are untouched.
stadtteil_report_review = reports_base[["report_id", "district"]].merge(
    stadtteil_label_review[["district_original", "proposed_stadtteil", "proposed_stadtteil_code"]],
    left_on="district", right_on="district_original", how="left", validate="many_to_one", sort=False
)
stadtteil_exception_mask = stadtteil_report_review["report_id"].isin(stadtteil_report_exceptions["report_id"])
stadtteil_report_review.loc[stadtteil_exception_mask, "proposed_stadtteil"] = "Unresolved"
stadtteil_report_review.loc[stadtteil_exception_mask, "proposed_stadtteil_code"] = ""
stadtteil_report_review["supported_proposal"] = stadtteil_report_review["proposed_stadtteil_code"].isin(stadtteil_codes.values())
stadtteil_proposal_counts = stadtteil_report_review.groupby("district")["supported_proposal"].sum()
stadtteil_label_review["supported_reports"] = stadtteil_labels.map(stadtteil_proposal_counts)
stadtteil_label_review["unresolved_reports"] = stadtteil_label_review["report_count"] - stadtteil_label_review["supported_reports"]


In [39]:
assert len(stadtteil_label_review) == 273 and stadtteil_label_review["district_original"].is_unique
assert set(stadtteil_label_review["district_original"]) == set(reports_base["district"])
assert stadtteil_label_review["report_count"].sum() == len(reports_base) == 5126
assert len(stadtteil_report_review) == 5126 and stadtteil_report_review["report_id"].is_unique
assert stadtteil_report_review["report_id"].tolist() == reports_base["report_id"].tolist()
assert stadtteil_report_review["district"].tolist() == reports_base["district"].tolist()
assert set(stadtteil_report_exceptions["report_id"]) == set(ortsteil_report_exceptions["report_id"])
assert stadtteil_report_review.loc[stadtteil_exception_mask, "proposed_stadtteil_code"].eq("").all()
assert stadtteil_label_review["reason"].notna().all() and stadtteil_label_review["evidence"].str.len().gt(0).all()
assert (stadtteil_label_review["supported_reports"] + stadtteil_label_review["unresolved_reports"]).equals(stadtteil_label_review["report_count"])
assert stadtteil_label_review.loc[stadtteil_label_review["review_category"].eq("Spelling candidate"), "supported_reports"].sum() == 0
assert stadtteil_label_review.loc[stadtteil_labels.isin(standalone_ortsteile), "proposed_stadtteil_code"].eq("").all()
assert stadtteil_label_review.loc[stadtteil_labels.eq("Oberneuland"), "proposed_stadtteil_code"].eq("36").all()

stadtteil_review_summary = stadtteil_label_review.groupby("review_category").agg(
    original_labels=("district_original", "size"), reports=("report_count", "sum"),
    supported_proposals=("supported_reports", "sum"), unresolved_reports=("unresolved_reports", "sum")
)
print(stadtteil_review_summary.to_string())
print("\nTotal labels:", len(stadtteil_label_review), "| Total reports:", stadtteil_label_review["report_count"].sum())
print("Preliminary Stadtteil coverage:", stadtteil_label_review["supported_reports"].sum())
print("Unresolved at Stadtteil level:", stadtteil_label_review["unresolved_reports"].sum())


                                original_labels  reports  supported_proposals  unresolved_reports
review_category                                                                                  
Ambiguous Mitte level                         1      119                    0                 119
Broad location                                6       14                    0                  14
Combined locations                           88       99                    0                  99
Direct Stadtteil name                         8      245                  245                   0
Landmark/local place                         31       68                    0                  68
Mixed report-level evidence                   2       56                   52                   4
Oberneuland special case                      1       44                   44                   0
Ortsteil to official parent                  69     2881                 2881                   0
Other municipality c

In [40]:
stadtteil_preview_columns = [
    "district_original", "report_count", "review_category",
    "proposed_stadtteil", "proposed_stadtteil_code", "supported_reports", "unresolved_reports"
]
print(stadtteil_label_review[stadtteil_preview_columns].head(8).to_string(index=False))
print("\nStandalone areas, Oberneuland and Mitte:")
print(stadtteil_label_review.loc[
    stadtteil_labels.isin(list(standalone_ortsteile) + ["Oberneuland", "Mitte"]),
    stadtteil_preview_columns
].to_string(index=False))
print("\nUnresolved report-specific exceptions:")
print(stadtteil_report_exceptions[["report_id", "district", "reason"]].to_string(index=False))


                           district_original  report_count              review_category proposed_stadtteil proposed_stadtteil_code  supported_reports  unresolved_reports
                                       Achim             1 Other municipality candidate         Unresolved                                          0                   1
                                  Achterdiek             1         Landmark/local place         Unresolved                                          0                   1
                               Alte Neustadt           101  Ortsteil to official parent           Neustadt                      21                101                   0
                     Alte Neustadt, Altstadt             1           Combined locations         Unresolved                                          0                   1
             Alte Neustadt, Bahnhofsvorstadt             1           Combined locations         Unresolved                                          0 

#### Internal geographic outcomes evidence preserved

The earlier tables remain the original preliminary evidence record across all 5,126 rows. Their supported proposals are reused conservatively. Spelling suggestions, unreviewed combined locations, landmarks and ambiguous cases remain unresolved. Exceptions override label-wide proposals.

Final internal outcomes use literal Unknown where no single location is supported. Both levels share inclusion flags. Part 2 and the shared column use **Stadtteile plus standalone Ortsteile**. Borgfeld, Blockland, Seehausen and Strom keep their names without invented parent relationships or codes. Oberneuland is Stadtteil 36; no Ortsteil is invented. Internal status columns distinguish missing information, unresolved evidence and standalone places.

In [41]:
ortsteil_working = reports[["report_id", "district"]].merge(
    ortsteil_label_review[["district_original", "review_category", "proposed_ortsteil"]],
    left_on="district", right_on="district_original", how="left", validate="many_to_one"
)
ortsteil_exception_mask = ortsteil_working["report_id"].isin(
    ortsteil_report_exceptions["report_id"]
)
ortsteil_working.loc[ortsteil_exception_mask, "proposed_ortsteil"] = "Unresolved"
ortsteil_working["supported_proposal"] = ~ortsteil_working["proposed_ortsteil"].isin(
    ["Unresolved", "Unknown"]
)
ortsteil_working["outcome_status"] = "unresolved"
ortsteil_working.loc[ortsteil_working["proposed_ortsteil"].eq("Unknown"), "outcome_status"] = "unknown"
ortsteil_working.loc[ortsteil_working["supported_proposal"], "outcome_status"] = "supported_proposal"

In [42]:
stadtteil_working = stadtteil_report_review.copy()
stadtteil_working["outcome_status"] = "unresolved"
stadtteil_working.loc[stadtteil_working["proposed_stadtteil"].eq("Unknown"), "outcome_status"] = "unknown"
stadtteil_working.loc[
    stadtteil_working["proposed_stadtteil"].eq("No parent Stadtteil"), "outcome_status"
] = "standalone_no_parent"
stadtteil_working.loc[stadtteil_working["supported_proposal"], "outcome_status"] = "supported_proposal"
for level, table in [("Ortsteile", ortsteil_working), ("Stadtteile", stadtteil_working)]:
    print(level)
    print(table["outcome_status"].value_counts().to_string())

Ortsteile
outcome_status
supported_proposal    2979
unresolved            1552
unknown                595
Stadtteile
outcome_status
supported_proposal      4118
unknown                  595
unresolved               380
standalone_no_parent      33


In [43]:
source_manifest = json.loads(
    (project_folder / "shared/source_manifest.json").read_text(encoding="utf-8")
)
source_columns = ["report_id"] + source_manifest["original_columns"]
assert reports_base.columns.tolist() == source_columns
assert reports.columns.tolist() == source_columns + ["include_in_analysis", "exclusion_reason"]
assert len(reports) == source_manifest["row_count"] == 5126
assert reports["report_id"].is_unique
assert reports["report_id"].equals(reports_base["report_id"])
assert pd.api.types.is_datetime64_any_dtype(reports["date_posted"])
assert reports["date_posted"].notna().all()
assert reports["date_posted"].dt.strftime("%Y-%m-%d").equals(reports_base["date_posted"])
assert reports["include_in_analysis"].dtype == bool
assert reports["include_in_analysis"].sum() == 5124
assert set(reports.loc[~reports["include_in_analysis"], "report_id"]) == set(duplicate_retained)
for excluded_id, retained_id in duplicate_retained.items():
    assert retained_id in reports.loc[reports["report_id"].eq(excluded_id), "exclusion_reason"].item()
assert reports["exclusion_reason"].ne("").equals(~reports["include_in_analysis"])
print("Rows, IDs, dates and inclusion flags: passed")

Rows, IDs, dates and inclusion flags: passed


In [44]:
# Reload independently so shared array references cannot hide a change.
source_check = pd.read_parquet(project_folder / "shared/reports_base.parquet")
embedding_columns = [name for name in source_columns if "embedding" in name]
plain_columns = [name for name in source_columns if name not in embedding_columns + ["date_posted"]]
pd.testing.assert_frame_equal(reports[plain_columns], source_check[plain_columns])
pd.testing.assert_frame_equal(reports_base[plain_columns], source_check[plain_columns])
assert reports_base["date_posted"].equals(source_check["date_posted"])
for name in embedding_columns:
    for working, base, source in zip(reports[name], reports_base[name], source_check[name]):
        assert np.array_equal(working, source) and np.array_equal(base, source)
        assert np.asarray(working).shape == (768,) and np.isfinite(working).all()
print("Original labels, text, URLs and both embedding columns: unchanged")

Original labels, text, URLs and both embedding columns: unchanged


In [45]:
exception_ids = set(ortsteil_report_exceptions["report_id"])
assert len(exception_ids) == 4
assert exception_ids == set(stadtteil_report_exceptions["report_id"])
for table, proposal, label_review in [
    (ortsteil_working, "proposed_ortsteil", ortsteil_label_review),
    (stadtteil_working, "proposed_stadtteil", stadtteil_label_review)
]:
    assert len(table) == 5126 and table["report_id"].is_unique
    assert table["report_id"].equals(reports["report_id"])
    assert table["district"].equals(reports["district"])
    assert table[proposal].notna().all() and table[proposal].map(type).eq(str).all()
    assert table.loc[table["report_id"].isin(exception_ids), proposal].eq("Unresolved").all()
    assert len(label_review) == 273 and label_review["district_original"].is_unique
    assert set(label_review["district_original"]) == set(reports["district"])
    assert table["supported_proposal"].sum() == label_review["supported_reports"].sum()
    held_labels = label_review.loc[label_review["review_category"].isin(
        ["Spelling candidate", "Combined locations"]), "district_original"]
    assert not table.loc[table["district"].isin(held_labels), "supported_proposal"].any()
print("Both levels: 5,126 unique outcomes; 273 labels; all four exceptions protected")

Both levels: 5,126 unique outcomes; 273 labels; all four exceptions protected


In [46]:
# Final internal outcomes stay separate from the original review evidence.
ortsteil_working["ortsteil_clean"] = ortsteil_working["proposed_ortsteil"].where(
    ortsteil_working["supported_proposal"], "Unknown"
)
stadtteil_working["stadtteil_clean"] = stadtteil_working["proposed_stadtteil"].where(
    stadtteil_working["supported_proposal"], "Unknown"
)
standalone_mask = stadtteil_working["district"].isin(standalone_ortsteile)
stadtteil_working.loc[standalone_mask, "stadtteil_clean"] = stadtteil_working.loc[standalone_mask, "district"]
inclusion_by_id = reports.set_index("report_id")["include_in_analysis"]
for table in [ortsteil_working, stadtteil_working]:
    table["include_in_analysis"] = table["report_id"].map(inclusion_by_id)

In [47]:
lookup_fields = ["district_original", "reason", "evidence", "official_reference", "reference_version"]
district_lookup = stadtteil_label_review[lookup_fields].copy()
district_lookup["district_clean"] = stadtteil_label_review["proposed_stadtteil"].where(
    stadtteil_label_review["proposed_stadtteil_code"].ne(""), "Unknown"
)
standalone_labels = district_lookup["district_original"].isin(standalone_ortsteile)
district_lookup.loc[standalone_labels, "district_clean"] = district_lookup.loc[standalone_labels, "district_original"]
district_lookup["decision"] = np.where(
    district_lookup["district_clean"].eq("Unknown"), "unresolved_to_Unknown", "reviewed_mapping"
)
district_lookup.loc[standalone_labels, "decision"] = "standalone_Ortsteil_no_parent"

In [48]:
# Notes preserve report-level overrides within the existing four-column schema.
lookup_notes = []
for row in district_lookup.to_dict("records"):
    exceptions = stadtteil_report_exceptions.loc[
        stadtteil_report_exceptions["district"].eq(row["district_original"])
    ]
    overrides = {
        entry["report_id"]: {"district_clean": "Unknown", "reason": entry["reason"], "url": entry["url"]}
        for entry in exceptions.to_dict("records")
    }
    lookup_notes.append(json.dumps({
        "representation": "Stadtteile plus standalone Ortsteile",
        "reason": row["reason"], "evidence": row["evidence"],
        "official_reference": row["official_reference"], "reference_version": row["reference_version"],
        "report_overrides": overrides
    }, ensure_ascii=False))
district_lookup["notes"] = lookup_notes
district_lookup = district_lookup[["district_original", "district_clean", "decision", "notes"]]
reports["district_clean"] = stadtteil_working["stadtteil_clean"].to_numpy()

In [49]:
coverage_rows = []
for level, table, field in [
    ("Ortsteile", ortsteil_working, "ortsteil_clean"),
    ("Stadtteile plus standalone Ortsteile", stadtteil_working, "stadtteil_clean")
]:
    assert len(table) == 5126 and table["report_id"].is_unique
    assert table["report_id"].equals(reports["report_id"])
    assert table["include_in_analysis"].equals(reports["include_in_analysis"])
    assert table[field].notna().all() and table[field].map(type).eq(str).all()
    assert table.loc[table["report_id"].isin(exception_ids), field].eq("Unknown").all()
    included = table.loc[table["include_in_analysis"]]
    coverage_rows.append([level, len(table), len(included),
                          int(included[field].ne("Unknown").sum()), int(included[field].eq("Unknown").sum())])
geographic_coverage = pd.DataFrame(
    coverage_rows, columns=["level", "all_rows", "included", "mapped_included", "unknown_included"]
)
assert reports.loc[reports["district_clean"].eq("Unknown"), "include_in_analysis"].all()
print(geographic_coverage.to_string(index=False))
print("Included standalone reports:", int((standalone_mask & reports["include_in_analysis"]).sum()))

                               level  all_rows  included  mapped_included  unknown_included
                           Ortsteile      5126      5124             2977              2147
Stadtteile plus standalone Ortsteile      5126      5124             4149               975
Included standalone reports: 33


**Explain your output:**

The original reviews and evidence are preserved. Supported proposals are now used conservatively; unverified candidates become Unknown. The four protected report IDs remain Unknown at both levels. Each report has one outcome per level, with identical inclusion flags.

The shared field uses **Stadtteile plus standalone Ortsteile**, retaining the four standalone names without invented parents. Ortsteil results remain separate. Earlier review totals describe all 5,126 source rows; the new coverage summary describes the 5,124 included reports. Unknown reports stay in citywide denominators. Exact-name mappings rely on recorded labels and hierarchy, not a fresh manual review of every report. Latest reference-version and future boundary checks remain limitations.

### 4. Publish the preparation handover

Save reports_prepared.parquet in your output folder, plus data_quality.csv, district_lookup.csv and duplicate_review.csv. Validate the report ID set equals the base set and original text/embeddings have not changed. Record the SHA-256 and tell Members 2–4 and 6 that the preparation version is ready. Freeze it before final model runs. Any change requires agreement and reruns.

In [50]:
import hashlib
import platform
from datetime import datetime, timezone
from importlib.metadata import version

contracts = json.loads((project_folder / "artifact_contracts.json").read_text(encoding="utf-8"))
preparation_schemas = contracts["roles"]["member_01"]["csv_schemas"]
prepared_columns = source_columns + ["district_clean", "include_in_analysis", "exclusion_reason"]
reports_prepared = reports[prepared_columns].copy()
representation = "Stadtteile plus standalone Ortsteile"
# Off by default. Authorized exports must not overwrite an existing handover.
write_preparation_files = False
preparation_files = [
    "reports_prepared.parquet", "data_quality.csv", "district_lookup.csv",
    "duplicate_review.csv", "run_metadata.json", "handover.md"
]

#### Local preparation handover review pending

The approved policies are applied below. All six preparation artifacts keep the required schemas. The label lookup contains defaults; its JSON notes explicitly list the four report-level overrides, which must be applied after label defaults when reconstructing the shared geography.

To reproduce, use the project Python 3.12.2 .venv and run setup and Sections 1–4 in order. The export switch defaults to False, so this rebuilds tables without writing files. The initial local export was authorized and enabled only in the execution context. The write guard refuses to overwrite existing files. A revised handover requires an intentional output/version plan and agreement before downstream reruns.

These are local review artifacts, not published or peer-approved results. Both geographic analyses are preserved. Final category-dependent tables and maps remain deferred.

In [62]:
quality_rows = [
    ["source_rows", len(reports_base), "All permanent IDs retained"],
    ["base_columns", len(source_columns), "Seven raw columns plus report_id"],
    ["prepared_columns", len(prepared_columns), "Exactly eleven contracted columns"],
    ["unique_report_ids", reports["report_id"].nunique(), "No repeated or regenerated IDs"],
    ["included_reports", int(reports["include_in_analysis"].sum()), "Approved inclusion policy"],
    ["excluded_duplicate_reports", int((~reports["include_in_analysis"]).sum()), "BR001569; BR002114"],
    ["original_district_labels", reports_base["district"].nunique(), "Original labels preserved"],
    ["protected_geographic_exceptions", len(exception_ids), "All Unknown and included"],
    ["publication_date_min", reports["date_posted"].min().strftime("%Y-%m-%d"), "Not incident date"],
    ["publication_date_max", reports["date_posted"].max().strftime("%Y-%m-%d"), "Partial boundary periods"]
]

In [63]:
for name in ["title_en", "description_en", "district"]:
    values = reports_base[name]
    quality_rows.extend([
        [name + "_missing", int(values.isna().sum()), "Source nulls"],
        [name + "_blank", int(values.str.strip().eq("").sum()), "Whitespace-only values"]
    ])
dates = reports_base["date_posted"]
missing_dates = dates.isna()
blank_dates = dates.str.strip().eq("").fillna(False)
parsed = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
quality_rows.extend([
    ["date_valid", int(parsed.notna().sum()), "YYYY-MM-DD"],
    ["date_missing", int(missing_dates.sum()), "Source nulls"],
    ["date_blank", int(blank_dates.sum()), "Source whitespace"],
    ["date_invalid", int((parsed.isna() & ~missing_dates & ~blank_dates).sum()), "Not missing or blank"]
])
for label in ["None", "null"]:
    quality_rows.append(["district_literal_" + label, int(reports_base["district"].eq(label).sum()), "Unknown; retained"])

In [64]:
for name in ["report_id", "url", "title_en", "description_en"]:
    values = reports_base[name]
    nonblank = values.notna() & values.str.strip().ne("")
    repeated = nonblank & values.duplicated(keep=False)
    quality_rows.extend([
        [name + "_repeated_groups", int(values[repeated].nunique()), "Exact nonblank source values"],
        [name + "_reports_in_repeated_groups", int(repeated.sum()), "Repetition alone is not an exclusion"]
    ])
repeated_pairs = reports_base.duplicated(["title_en", "description_en"], keep=False)
quality_rows.extend([
    ["repeated_title_description_pairs", len(reports_base.loc[repeated_pairs, ["title_en", "description_en"]].drop_duplicates()), "Exact pairs"],
    ["reports_in_repeated_text_pairs", int(repeated_pairs.sum()), "Pairs checked across five fields"]
])
duplicate_ids = list(duplicate_retained) + list(duplicate_retained.values())
duplicate_review = reports_base.loc[reports_base["report_id"].isin(duplicate_ids), ["report_id", "url"]].copy()
duplicate_review["decision"] = "retain"
duplicate_review["reason"] = ""
for excluded_id, retained_id in duplicate_retained.items():
    duplicate_review.loc[duplicate_review["report_id"].eq(excluded_id), ["decision", "reason"]] = [
        "exclude", "Duplicate of retained " + retained_id + "; date, title, description, URL and district match."
    ]
    duplicate_review.loc[duplicate_review["report_id"].eq(retained_id), "reason"] = (
        "Retained matching report; " + excluded_id + " remains in the dataset but is excluded."
    )

In [54]:
for name in embedding_columns:
    vectors = [np.asarray(vector) for vector in reports_base[name]]
    assert all(v.shape == (768,) and np.issubdtype(v.dtype, np.number) and np.isfinite(v).all() for v in vectors)
    quality_rows.append([name + "_valid_768_finite", len(vectors), "Numeric, finite, unchanged and aligned by ID"])
for row in geographic_coverage.to_dict("records"):
    for metric in ["mapped_included", "unknown_included"]:
        quality_rows.append([row["level"] + "_" + metric, row[metric], "Included cohort; levels are never added"])
quality_rows.extend([
    ["shared_mapped_all_rows", int(reports["district_clean"].ne("Unknown").sum()), representation],
    ["shared_unknown_all_rows", int(reports["district_clean"].eq("Unknown").sum()), "Unresolved or missing information"],
    ["standalone_included", int((standalone_mask & reports["include_in_analysis"]).sum()), "Known Ortsteile; no parent"],
    ["source_content_preserved", True, "Text, district, URLs and both vectors checked against source"],
    ["peer_review", "pending", "Reviewer not assigned; no approval claimed"]
])
data_quality = pd.DataFrame(quality_rows, columns=preparation_schemas["data_quality.csv"])

In [56]:
metadata_template = json.loads(
    (project_folder / "templates/member_01/run_metadata.json").read_text(encoding="utf-8")
)
run_metadata = dict(metadata_template)
package_names = [
    line.strip() for line in (project_folder / "requirements.txt").read_text().splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]
run_metadata.update({
    "author": "Meriam El Askri", "reviewer": "", "status": "in_progress",
    "run_date": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "package_versions": {name: version(name) for name in package_names},
    "methods": ["Strict publication-date parsing", "Conservative reviewed geography", "Two approved duplicate flags"],
    "parameters": {"representation": representation, "unknown": "Unknown",
                   "duplicate_retained": duplicate_retained, "geographic_exceptions": sorted(exception_ids),
                   "date_format": "%Y-%m-%d", "execution": "Project .venv Python; local preparation cells"},
    "counts": {"source_rows": 5126, "included": 5124, "excluded": 2,
               "geographic_coverage": geographic_coverage.to_dict("records")},
    "notes": ["Local review artifacts; not published or peer approved.",
              "Reviewer not assigned. Final category analyses await Member 06.",
              "Package versions describe installed distributions; no models were executed.",
              "Latest official reference version and future boundary compatibility remain unverified."]
})
assert set(run_metadata) == set(metadata_template)

PackageNotFoundError: No package metadata was found for umap-learn

In [57]:
for table, filename in [
    (data_quality, "data_quality.csv"), (district_lookup, "district_lookup.csv"),
    (duplicate_review, "duplicate_review.csv")
]:
    assert table.columns.tolist() == preparation_schemas[filename]
    assert table.columns.tolist() == pd.read_csv(project_folder / "templates/member_01" / filename).columns.tolist()
assert len(district_lookup) == 273 and district_lookup["district_original"].is_unique
assert len(duplicate_review) == 4 and duplicate_review["report_id"].is_unique
assert reports_prepared.columns.tolist() == prepared_columns
assert reports_prepared["include_in_analysis"].dtype == bool
assert reports_prepared["report_id"].equals(reports_base["report_id"])
assert reports_prepared["district_clean"].notna().all()
# Reconstruct label defaults, then protect the documented report-level exceptions.
rebuilt = reports_base["district"].map(district_lookup.set_index("district_original")["district_clean"])
for notes in district_lookup["notes"]:
    for report_id, override in json.loads(notes)["report_overrides"].items():
        rebuilt.loc[reports_base["report_id"].eq(report_id)] = override["district_clean"]
assert rebuilt.equals(reports_prepared["district_clean"])
print("Contract schemas and lookup reconstruction: passed")

Contract schemas and lookup reconstruction: passed


In [58]:
for filename, checksum_key in [
    (source_manifest["raw_file"], "raw_sha256"), (source_manifest["base_file"], "base_sha256")
]:
    checksum = hashlib.sha256((project_folder / filename).read_bytes()).hexdigest()
    assert checksum == source_manifest[checksum_key]
assert run_metadata["source_sha256"] == source_manifest["raw_sha256"]
assert run_metadata["base_sha256"] == source_manifest["base_sha256"]
print("Raw and base SHA-256 match the manifest")

Raw and base SHA-256 match the manifest


In [59]:
if write_preparation_files:
    assert not any((output_folder / name).exists() for name in preparation_files), "Existing handover: do not overwrite."
    reports_prepared.to_parquet(output_folder / "reports_prepared.parquet", index=False)
    for table, filename in [
        (data_quality, "data_quality.csv"), (district_lookup, "district_lookup.csv"),
        (duplicate_review, "duplicate_review.csv")
    ]:
        table.to_csv(output_folder / filename, index=False, encoding="utf-8")
    prepared_sha256 = hashlib.sha256((output_folder / "reports_prepared.parquet").read_bytes()).hexdigest()
    run_metadata["prepared_sha256"] = prepared_sha256
    print("Prepared file SHA-256:", prepared_sha256)
else:
    print("Export disabled: preparation rebuilt in memory only.")

Export disabled: preparation rebuilt in memory only.


In [65]:
handover_template = (project_folder / "templates/member_01/handover.md").read_text(encoding="utf-8")
handover_sections = {
    "What I completed": "Prepared all 5,126 rows with 5,124 included and two duplicate flags. Both geographic reviews and internal outcomes remain in the notebook. Final category-dependent analyses are not complete.",
    "Input files and inclusion rules": (
        "Immutable raw/base inputs match source_manifest.json. Dates are publication dates, not incident dates. "
        "BR001569 is excluded as a match of retained BR001568; BR002114 is excluded as a match of retained BR002113. "
        "All other rows remain included. Original IDs, labels, text, URLs and both 768-dimensional embeddings remain aligned and unchanged."
    ),
    "Output files and row counts": (
        "reports_prepared.parquet: 5,126 rows, 11 columns.\n"
        "data_quality.csv: " + str(len(data_quality)) + " checks.\n"
        "district_lookup.csv: 273 original-label defaults, with JSON report_overrides in notes.\n"
        "duplicate_review.csv: four report decisions.\nrun_metadata.json and handover.md: metadata and instructions.\n"
        "Final district_category_counts.csv and highest_district_by_category.csv await approved categories."
    ),
    "Methods and parameter choices": (
        "Shared representation: Stadtteile plus standalone Ortsteile. Borgfeld, Blockland, Seehausen and Strom retain their official names without invented parents or codes. "
        "Oberneuland uses its reviewed Stadtteil identity; no Ortsteil is invented. Unverified spelling candidates, combined labels, landmarks, ambiguous and other-municipality candidates remain Unknown. "
        "BR001277, BR003479, BR003518 and BR001797 remain Unknown and included. Apply JSON report_overrides from lookup notes after label defaults; a label must never override these exceptions.\n\n"
        "To reproduce: select the project Python 3.12.2 .venv and run setup plus Sections 1–4 in order. The export switch defaults to False; existing artifacts are never overwritten. "
        "No network calls, geocoding or new embeddings are required. Revised preparation needs agreement and dependent reruns."
    ),
    "Findings supported by saved outputs": geographic_coverage.to_string(index=False) + "\n\nBoth levels count reports once with identical inclusion flags. Never add their totals together. Unknown stays in citywide denominators.",
    "Validation and reviewer comments": (
        "Source rows/IDs/content, finite vectors, dates, flags, schemas and report-level geography validated. "
        "The export validation cell checks serialized files, inventory and checksum. Reviewer not assigned; peer review pending. No acceptance or publication is claimed."
    ),
    "Limitations and unresolved questions": (
        "Conservative geography partly relies on reviewed exact names, not a fresh individual review of every report. Reports can be updates, appeals or cover several incidents; counts are reports, not confirmed crimes or risk rates. "
        "Unknown retains missing/unresolved information. Standalone names are known locations, not missing parent relationships. "
        "References: indexed Bremen street directory November 2025, cross-checked with Die Ortsteile der Stadt Bremen 2015 (published May 2016); latest version and map-boundary compatibility remain unverified. "
        "Official URLs and report evidence remain in Section 3 and district_lookup.csv. Package versions do not establish successful model runs. "
        "Notebook kernel metadata is inherited; execution here used the project .venv Python, not a verified editor kernel session."
    ),
    "Where these results belong in the main notebook": (
        "Use Data preparation now; district distribution and citywide composition follow approved categories later. The Main Working Station must check full ID coverage, source preservation and exact prepared checksum before accepting this local handover.\n\n"
        "Members 02–04: verify SHA-256 against run_metadata.json, then load outputs/member_01/reports_prepared.parquet with pandas.read_parquet. "
        "Select eligible = prepared.loc[prepared['include_in_analysis']].copy(). Build text and embedding inputs from that SAME frame/order; never sort separately. "
        "Keep report_id strings. Assignment exports must still cover all 5,126 IDs, with assignment_status='excluded' for the two excluded rows. "
        "Join using report_id and validate='one_to_one', never URL. Record the exact prepared-file SHA-256 in each run_metadata.json.\n\n"
        "Member 06: use the same prepared version. Approved report_categories.csv must cover all IDs, with selected_model.json and matching input checksums. "
        "Later join categories one-to-one for BOTH internal analyses; retain identical inclusion flags and Unknown in denominators. Do not invent categories. "
        "Any preparation change requires version agreement and rebuilding dependent results."
    )
}
handover_text = handover_template.replace(
    "Status: NOT STARTED — this is a template, not a submitted result.",
    "Status: LOCAL PREPARATION FOR REVIEW — peer review and publication pending."
)
for heading, body in handover_sections.items():
    handover_text = handover_text.replace("## " + heading, "## " + heading + "\n\n" + body + "\n")
handover_text = handover_text.replace("Name:\n", "Name: Meriam El Askri\n")
handover_text = handover_text.replace("Reviewer:\n", "Reviewer: Not assigned; pending\n")
handover_text = handover_text.replace("Source/base SHA-256:\n", "Source/base SHA-256: " + run_metadata["source_sha256"] + " / " + run_metadata["base_sha256"] + "\n")
handover_text = handover_text.replace("Final categories SHA-256 (if used):", "Final categories SHA-256 (if used): pending; not used")
handover_text = handover_text.replace("Python/package versions:", "Python/package versions: " + run_metadata["python_version"] + "; exact distributions in run_metadata.json.")
if write_preparation_files:
    handover_text = handover_text.replace("Prepared dataset SHA-256:", "Prepared dataset SHA-256: " + prepared_sha256)
    (output_folder / "run_metadata.json").write_text(json.dumps(run_metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    (output_folder / "handover.md").write_text(handover_text, encoding="utf-8")

In [ ]:
if write_preparation_files:
    saved = pd.read_parquet(output_folder / "reports_prepared.parquet")
    pd.testing.assert_frame_equal(saved, reports_prepared)
    assert len(saved) == 5126 and saved["report_id"].is_unique
    assert saved["include_in_analysis"].dtype == bool and saved["include_in_analysis"].sum() == 5124
    assert set(saved.loc[~saved["include_in_analysis"], "report_id"]) == set(duplicate_retained)
    assert saved.loc[saved["report_id"].isin(exception_ids), "district_clean"].eq("Unknown").all()
    assert saved.loc[saved["district_clean"].eq("Unknown"), "include_in_analysis"].all()
    for table, filename in [
        (data_quality, "data_quality.csv"), (district_lookup, "district_lookup.csv"),
        (duplicate_review, "duplicate_review.csv")
    ]:
        stored = pd.read_csv(output_folder / filename, dtype=str, keep_default_na=False)
        pd.testing.assert_frame_equal(stored, table.astype(str).reset_index(drop=True))
    saved_metadata = json.loads((output_folder / "run_metadata.json").read_text(encoding="utf-8"))
    actual_sha = hashlib.sha256((output_folder / "reports_prepared.parquet").read_bytes()).hexdigest()
    assert saved_metadata["prepared_sha256"] == actual_sha
    assert actual_sha in (output_folder / "handover.md").read_text(encoding="utf-8")
    assert set(saved_metadata) == set(metadata_template)
    assert {file.name for file in output_folder.iterdir() if file.name != ".gitkeep"} == set(preparation_files)
    print("Read-back validation: passed; six-file inventory and SHA-256 verified")
    print("Prepared bytes:", (output_folder / "reports_prepared.parquet").stat().st_size)

Read-back validation: passed; six-file inventory and SHA-256 verified
Prepared bytes: 66527577


**Explain your output:**

The six local preparation artifacts have been created and validated for review, without publication. All 5,126 rows remain, 5,124 are included and exactly two matching records are excluded. Original content and embedding alignment are preserved, dates are datetime, and the shared field uses **Stadtteile plus standalone Ortsteile**. The actual prepared-file checksum is recorded in metadata and handover.

Members 02–04 must use this exact version, respect inclusion flags and preserve ID alignment. Their assignment files still cover all IDs, including excluded statuses. The Main Working Station must verify the common checksum before integration. Preparation changes require agreement and dependent reruns.

Both geographic analyses remain. Member 06's approved report categories and selected model are later dependencies, joined one-to-one by report ID with version checks. No final category results were generated. Peer reviewer assignment, map-boundary checks and publication remain pending; technical validation is not peer approval.

### 5. Wait for approved categories, then join them

Load outputs/member_06/report_categories.csv and selected_model.json only after group approval. Join using report_id and validate="one_to_one". Confirm category rows cover every base ID, input checksums match, and eligibility flags are respected. District charts must use the approved category column, not an experimental topic ID.

In [67]:
import io

EXPECTED_PREPARED_SHA256 = "bf64238b7f929551cf20dadcdd314fa8bf17efe2cd7eab6174fd44dab1a1c139"
# Reset state on every run so an earlier successful run cannot supply stale results.
category_ready = False
analysis_ready = False
analysis_input_hashes = {}
category_errors = []
category_inputs = {}
category_joined = None
analysis_results = {}
district_category_counts = None
highest_district_by_category = None
eligible_categories = None
geographic_levels = {}
counts = composition = leaders = known_counts = None
area_totals = category_totals = maximum = None
result = results = standard = None
totals = heatmap = image = fig = ax = period = figure_path = None
approved_categories = category_mapping = selected_mapping = None
mapped_rows = mapping_check = manual_rows = overrides = override_check = None
category_prepared = geography_for_categories = geography_check = shared_check = None
category_input_hashes = {}
write_category_csvs = False
create_category_figures = False
save_category_figures = False

def require(condition, message):
    if not condition:
        raise ValueError(message)

member06_folder = project_folder / "outputs/member_06"
category_paths = {
    "prepared": project_folder / "outputs/member_01/reports_prepared.parquet",
    "preparation_metadata": project_folder / "outputs/member_01/run_metadata.json",
    "categories": member06_folder / "report_categories.csv",
    "selection": member06_folder / "selected_model.json",
    "mapping": member06_folder / "category_mapping.csv",
    "metadata": member06_folder / "run_metadata.json"
}

In [68]:
# Read local bytes once. Every subsequent parse uses this same input snapshot.
try:
    for name, path in category_paths.items():
        if path.is_file():
            category_inputs[name] = path.read_bytes()
        else:
            category_errors.append("Missing required input: " + str(path.relative_to(project_folder)))
    if "prepared" in category_inputs:
        require(hashlib.sha256(category_inputs["prepared"]).hexdigest() == EXPECTED_PREPARED_SHA256,
                "Prepared Parquet SHA-256 differs from the accepted version.")
except (OSError, ValueError) as error:
    category_errors.append(str(error))

In [69]:
if not category_errors:
    try:
        selection = json.loads(category_inputs["selection"])
        category_metadata = json.loads(category_inputs["metadata"])
        preparation_metadata = json.loads(category_inputs["preparation_metadata"])
        require(isinstance(selection, dict) and isinstance(category_metadata, dict), "Selection/metadata must be JSON objects.")
        require(category_metadata.get("member_id") == "member_06", "Metadata is not Member 06's handover.")
        require(selection.get("status") == "approved", "Model selection is missing approval or is provisional.")
        model_id = selection.get("model_id")
        require(model_id in contracts["model_ids"], "Selected model_id is not one of the four contracted models.")
        approvers = selection.get("approvers")
        require(isinstance(approvers, list) and bool(approvers) and
                all(isinstance(name, str) and name.strip() for name in approvers), "Approval names are missing.")
        require(isinstance(selection.get("selection_reason"), str) and bool(selection["selection_reason"].strip()),
                "Model selection reason is missing.")
        for metadata in [selection, category_metadata, preparation_metadata]:
            require(metadata.get("prepared_sha256") == EXPECTED_PREPARED_SHA256, "Prepared checksum mismatch in selection or metadata.")
        for key in ["source_sha256", "base_sha256"]:
            require(category_metadata.get(key) == preparation_metadata.get(key) == run_metadata[key],
                    "Member 06 source/base version differs: " + key)
        for name, key in [("mapping", "category_mapping_sha256"), ("categories", "report_categories_sha256")]:
            require(selection.get(key) == hashlib.sha256(category_inputs[name]).hexdigest(), "Selected " + key + " is missing or incorrect.")
        require(category_metadata.get("report_categories_sha256") == selection["report_categories_sha256"],
                "Member 06 metadata category checksum differs from selected_model.json.")
    except (ValueError, TypeError, OverflowError, KeyError, AttributeError) as error:
        category_errors.append("Approval/version check: " + str(error))

In [70]:
if not category_errors:
    try:
        approved_categories = pd.read_csv(io.BytesIO(category_inputs["categories"]), dtype=str, keep_default_na=False)
        category_mapping = pd.read_csv(io.BytesIO(category_inputs["mapping"]), dtype=str, keep_default_na=False)
        category_schemas = contracts["roles"]["member_06"]["csv_schemas"]
        for table, filename in [(approved_categories, "report_categories.csv"), (category_mapping, "category_mapping.csv")]:
            require(table.columns.tolist() == category_schemas[filename], "Incorrect schema: " + filename)
        require(len(approved_categories) == 5126 and approved_categories["report_id"].is_unique,
                "Categories must contain exactly 5,126 unique report IDs.")
        require(set(approved_categories["report_id"]) == set(reports_base["report_id"]), "Category report IDs differ from the base IDs.")
        require(approved_categories["selected_model_id"].eq(model_id).all(), "Category rows use a different selected model.")
        require(approved_categories["category_status"].isin(contracts["category_status_values"]).all(), "Invalid category_status.")
        require(approved_categories["crime_category"].eq(approved_categories["crime_category"].str.strip()).all(),
                "Category names contain surrounding whitespace.")
        require(category_mapping["model_id"].isin(contracts["model_ids"]).all(), "Mapping contains an unknown model.")
        require(set(category_mapping["model_id"]) == set(contracts["model_ids"]), "Mapping must cover all four compared models.")
        require(category_mapping[["crime_category", "decision", "reason"]].apply(lambda col: col.str.strip().ne("")).all().all(),
                "Mapping contains blank categories, decisions or reasons.")
    except (ValueError, TypeError, OverflowError, KeyError, AttributeError, pd.errors.ParserError) as error:
        category_errors.append("Schema/ID check: " + str(error))

In [71]:
if not category_errors:
    try:
        for table in [approved_categories, category_mapping]:
            numeric_topics = pd.to_numeric(table["topic_id"].replace("", pd.NA), errors="coerce")
            require(numeric_topics.notna().eq(table["topic_id"].ne("")).all(), "A topic_id is not numeric.")
            require(numeric_topics.dropna().mod(1).eq(0).all(), "A topic_id is not an integer.")
            table["topic_id"] = numeric_topics.astype("Int64")
        require(category_mapping["topic_id"].notna().all(), "Mapping topic IDs cannot be blank.")
        require(not category_mapping.duplicated(["model_id", "topic_id"]).any(), "Repeated model/topic keys in category mapping.")
        selected_mapping = category_mapping.loc[category_mapping["model_id"].eq(model_id)]
        mapped_rows = approved_categories.loc[approved_categories["category_status"].eq("mapped_topic")]
        require(mapped_rows["topic_id"].notna().all() and mapped_rows["topic_id"].ge(0).all(),
                "mapped_topic requires a nonnegative topic ID; noise is not an approved mapped topic.")
        mapping_check = mapped_rows.merge(
            selected_mapping[["topic_id", "crime_category"]], on="topic_id",
            how="left", validate="many_to_one", suffixes=("", "_mapping"), indicator=True
        )
        require(mapping_check["_merge"].eq("both").all(), "A mapped topic is absent from the selected mapping.")
        require(mapping_check["crime_category"].eq(mapping_check["crime_category_mapping"]).all(),
                "Category assignments contradict the approved topic mapping.")
    except (ValueError, TypeError, OverflowError, KeyError, pd.errors.MergeError) as error:
        category_errors.append("Topic/mapping check: " + str(error))

In [72]:
if not category_errors:
    try:
        manual_rows = approved_categories.loc[approved_categories["category_status"].eq("manual_override")]
        if not manual_rows.empty:
            override_path = member06_folder / "category_overrides.csv"
            require(override_path.is_file(), "Manual category overrides require category_overrides.csv.")
            category_paths["overrides"] = override_path
            category_inputs["overrides"] = override_path.read_bytes()
            overrides = pd.read_csv(io.BytesIO(category_inputs["overrides"]), dtype=str, keep_default_na=False)
            require(overrides.columns.tolist() == ["report_id", "crime_category", "reason", "reviewer", "category_status"],
                    "Incorrect category_overrides.csv schema.")
            require(overrides["report_id"].is_unique and set(overrides["report_id"]) == set(manual_rows["report_id"]),
                    "Manual override IDs do not exactly match manual_override rows.")
            require(overrides["category_status"].eq("manual_override").all(), "Invalid manual override status.")
            require(overrides[["reason", "reviewer"]].apply(lambda col: col.str.strip().ne("")).all().all(),
                    "Manual overrides need reasons and reviewers.")
            override_check = manual_rows.merge(overrides, on="report_id", validate="one_to_one", suffixes=("", "_override"))
            require(override_check["crime_category"].eq(override_check["crime_category_override"]).all(),
                    "Manual override categories differ from final categories.")
    except (OSError, ValueError, TypeError, KeyError, pd.errors.ParserError) as error:
        category_errors.append("Manual override check: " + str(error))

In [73]:
if not category_errors:
    try:
        category_prepared = pd.read_parquet(io.BytesIO(category_inputs["prepared"]))
        require(category_prepared.columns.tolist() == prepared_columns, "Prepared schema is incorrect.")
        require(len(category_prepared) == 5126 and category_prepared["report_id"].is_unique, "Prepared IDs are not unique/full coverage.")
        require(category_prepared["report_id"].equals(reports_base["report_id"]), "Prepared IDs/order differ from source.")
        require(category_prepared["include_in_analysis"].dtype == bool and
                category_prepared["include_in_analysis"].sum() == 5124, "Incorrect inclusion flags.")
        require(set(category_prepared.loc[~category_prepared["include_in_analysis"], "report_id"]) ==
                {"BR001569", "BR002114"}, "Incorrect duplicate exclusions.")
        category_joined = category_prepared[
            ["report_id", "date_posted", "district_clean", "include_in_analysis"]
        ].merge(approved_categories, on="report_id", how="left", validate="one_to_one", indicator=True)
        require(category_joined["_merge"].eq("both").all(), "Not every prepared report joined to a category.")
        excluded = ~category_joined["include_in_analysis"]
        require(category_joined["category_status"].eq("excluded").equals(excluded), "Category exclusion statuses differ from preparation.")
        require(category_joined.loc[~excluded, "crime_category"].str.strip().ne("").all(),
                "Included reports need explicit categories, including Unknown/Unclear when applicable.")
    except (OSError, ValueError, TypeError, KeyError, pd.errors.MergeError) as error:
        category_errors.append("Prepared/join/exclusion check: " + str(error))

In [74]:
if not category_errors:
    try:
        geography_for_categories = ortsteil_working[["report_id", "ortsteil_clean", "include_in_analysis"]].copy()
        require(geography_for_categories["report_id"].is_unique and len(geography_for_categories) == 5126,
                "Ortsteil working outcomes are not one per report.")
        geography_check = category_joined.merge(
            geography_for_categories, on="report_id", validate="one_to_one", suffixes=("", "_ortsteil")
        )
        require(geography_check["include_in_analysis"].equals(geography_check["include_in_analysis_ortsteil"]),
                "Geographic levels have different inclusion flags.")
        shared_check = category_joined[["report_id", "district_clean"]].merge(
            stadtteil_working[["report_id", "stadtteil_clean"]], on="report_id", validate="one_to_one"
        )
        require(shared_check["district_clean"].equals(shared_check["stadtteil_clean"]),
                "Internal Stadtteil outcomes differ from the accepted prepared geography.")
        require(geography_check[["ortsteil_clean", "district_clean"]].notna().all().all(), "Missing geographic outcome.")
        protected = geography_check["report_id"].isin(["BR001277", "BR003479", "BR003518", "BR001797"])
        require(geography_check.loc[protected, ["ortsteil_clean", "district_clean"]].eq("Unknown").all().all(),
                "A protected geographic exception was overwritten.")
        category_joined = geography_check.drop(columns=["_merge", "include_in_analysis_ortsteil"])
        category_input_hashes = {name: hashlib.sha256(data).hexdigest() for name, data in category_inputs.items()}
        category_ready = True
    except (ValueError, TypeError, OverflowError, KeyError, pd.errors.MergeError) as error:
        category_errors.append("Geographic join check: " + str(error))
if not category_ready:
    category_joined = None
    print("Category-dependent work skipped:")
    for reason in category_errors:
        print("-", reason)
else:
    print("Approved categories validated; both analyses use the same 5,124 included reports.")

Category-dependent work skipped:
- Missing required input: outputs\member_06\report_categories.csv
- Missing required input: outputs\member_06\selected_model.json
- Missing required input: outputs\member_06\category_mapping.csv


**Methods and approval dependency**

Section 5 reads only local handovers. The accepted prepared-file SHA-256 is pinned, and selection must be approved with named approvers and a reason. The exact category and mapping bytes must match selected_model.json; Member 06 metadata must identify the same prepared and category versions. Mapping covers all four candidate models; mapped-topic rows must agree with the selected mapping. Manual overrides require their documented reasons and reviewers.

All 5,126 IDs must join one-to-one. Exactly the two prepared duplicate exclusions must have excluded category status. Every included row needs an explicit category; Unknown, Unclear and non-crime labels supplied by the approved handover are retained rather than replaced or dropped. Invalid or missing inputs skip Sections 6–7 with the failed condition. Approval is checked as recorded metadata, not independently authenticated.

No approval or category content is invented. Member 06 must supply report_categories.csv, selected_model.json, category_mapping.csv and run_metadata.json, plus category_overrides.csv when used. The six existing preparation artifacts remain untouched.

### 6. Count categories by district

Use pandas groupby and size. Export district_category_counts.csv. Calculate share_within_district (category count / all included reports in that district) and share_of_category_reports (district count / all included reports of that category citywide, with Unknown in the denominator). Explain both denominators. Show counts and composition, and include ties in highest_district_by_category.csv.

In [75]:
# Starting analysis also invalidates any previous complete or partial results.
analysis_ready = False
analysis_input_hashes = {}
analysis_results = {}
district_category_counts = None
highest_district_by_category = None
eligible_categories = None
geographic_levels = {}
if category_ready:
    try:
        eligible_categories = category_joined.loc[category_joined["include_in_analysis"]].copy()
        require(len(eligible_categories) == 5124 and eligible_categories["report_id"].is_unique,
                "Eligible category cohort is not exactly 5,124 unique reports.")
        geographic_levels = {
            "Part 1: Ortsteile": "ortsteil_clean",
            "Part 2: Stadtteile plus standalone Ortsteile": "district_clean"
        }
    except Exception as error:
        category_ready = False
        eligible_categories = None
        category_errors.append("Analysis setup failed: " + str(error))
        print(category_errors[-1])
else:
    print("Both geographic analyses skipped: approved category validation has not passed.")

Both geographic analyses skipped: approved category validation has not passed.


In [76]:
analysis_ready = False
analysis_input_hashes = {}
analysis_results = {}
district_category_counts = None
highest_district_by_category = None
if category_ready:
    try:
        for level, area_column in geographic_levels.items():
            counts = eligible_categories.groupby(
                [area_column, "crime_category"], dropna=False
            ).size().rename("report_count").reset_index().rename(columns={area_column: "district_clean"})
            area_totals = counts.groupby("district_clean")["report_count"].transform("sum")
            category_totals = counts.groupby("crime_category")["report_count"].transform("sum")
            counts["share_within_district"] = counts["report_count"] / area_totals
            counts["share_of_category_reports"] = counts["report_count"] / category_totals
            composition = eligible_categories.groupby("crime_category").size().rename("report_count").reset_index()
            composition["share_citywide"] = composition["report_count"] / len(eligible_categories)
            # Unknown is a coverage outcome, not a named geographic area to rank.
            known_counts = counts.loc[counts["district_clean"].ne("Unknown")]
            maximum = known_counts.groupby("crime_category")["report_count"].transform("max")
            leaders = known_counts.loc[known_counts["report_count"].eq(maximum),
                                       ["crime_category", "district_clean", "report_count"]].copy()
            analysis_results[level] = {"counts": counts, "citywide": composition, "leaders": leaders}
    except Exception as error:
        category_ready = False
        category_errors.append("Aggregation failed: " + str(error))
        analysis_results = {}
        print(category_errors[-1])

In [77]:
analysis_ready = False
analysis_input_hashes = {}
district_category_counts = None
highest_district_by_category = None
if category_ready:
    try:
        require(set(analysis_results) == {
            "Part 1: Ortsteile", "Part 2: Stadtteile plus standalone Ortsteile"
        }, "Both geographic analyses must be complete before outputs are enabled.")
        for level, result in analysis_results.items():
            counts = result["counts"]
            require(counts["report_count"].sum() == 5124, level + ": counts do not reconcile.")
            require(not counts.duplicated(["district_clean", "crime_category"]).any(), level + ": repeated area/category cells.")
            require(np.allclose(counts.groupby("district_clean")["share_within_district"].sum(), 1),
                    level + ": within-area shares do not sum to one.")
            require(np.allclose(counts.groupby("crime_category")["share_of_category_reports"].sum(), 1),
                    level + ": category shares do not sum to one including Unknown geography.")
            require(np.isclose(result["citywide"]["share_citywide"].sum(), 1), "Citywide composition does not sum to one.")
            require(counts[["share_within_district", "share_of_category_reports"]].ge(0).all().all() and
                    counts[["share_within_district", "share_of_category_reports"]].le(1).all().all(), "Shares outside [0,1].")
        results = list(analysis_results.values())
        pd.testing.assert_frame_equal(results[0]["citywide"], results[1]["citywide"])
        standard = analysis_results["Part 2: Stadtteile plus standalone Ortsteile"]
        district_category_counts = standard["counts"][preparation_schemas["district_category_counts.csv"]]
        highest_district_by_category = standard["leaders"][preparation_schemas["highest_district_by_category.csv"]]
        for level, result in analysis_results.items():
            print(level, "| included reports: 5,124; categories:", len(result["citywide"]))
            display(result["citywide"].head(20))
        analysis_input_hashes = dict(category_input_hashes)
        analysis_ready = True
    except Exception as error:
        category_ready = False
        category_errors.append("Result validation failed: " + str(error))
        analysis_results = {}
        district_category_counts = highest_district_by_category = None
        print(category_errors[-1])

In [78]:
# Recheck files at output time; rerun Sections 5–7 if any input changed.
def category_inputs_unchanged():
    global analysis_ready
    if not category_ready or analysis_input_hashes != category_input_hashes:
        analysis_ready = False
        return False
    try:
        unchanged = all(
            path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == category_input_hashes[name]
            for name, path in category_paths.items()
        )
    except (OSError, KeyError):
        unchanged = False
    if not unchanged:
        analysis_ready = False
    return unchanged

if not analysis_ready or not category_ready or not category_inputs_unchanged():
    print("Final CSVs disabled: complete analysis and approved, unchanged input versions are required.")
elif not write_category_csvs:
    print("Validated tables remain in memory; final CSV export switch is False.")
else:
    targets = [output_folder / "district_category_counts.csv", output_folder / "highest_district_by_category.csv"]
    require(not any(path.exists() for path in targets), "Existing category CSVs will not be overwritten.")
    district_category_counts.to_csv(targets[0], index=False, encoding="utf-8")
    highest_district_by_category.to_csv(targets[1], index=False, encoding="utf-8")

Final CSVs disabled: complete analysis and approved, unchanged input versions are required.


**Methods, denominators and interpretation**

Both parts use the same 5,124 included report IDs and the same approved category assignments. Each report contributes once per geographic analysis. Do not add the two levels together. Unknown geography and approved Unknown/Unclear categories remain visible.

Within-area category share is category reports divided by all included reports in that area. An area's share of a category divides by that category's included citywide total, including Unknown geography. Citywide category composition divides by all 5,124 included reports and is identical in both analyses. Shares are fractions from 0 to 1.

Highest-count named areas retain every tie. Unknown geography stays in counts and all denominators but is not ranked as a named district. A category with no known geographic assignment has no named-area leader. Interpret this as missing geographic coverage, not zero citywide reports.

Only Part 2 supplies the two required standard CSVs, using their unchanged schemas and shared district_clean meaning. Part 1 tables and citywide composition remain internal; no extra tabular handover is introduced. The CSV switch remains False until a later authorized run. No numerical category findings are claimed while inputs are pending.

### 7. Map and interpret the distribution

Produce a readable count chart, a district/category heatmap, and a geographic map using suitable boundary data if available. The source has no coordinates: a scatterplot is not a district map. Record the boundary source/license and unmatched labels. Coordinate with the group to obtain compatible official boundaries; raise missing boundary access before final submission. Never label a count hotspot as a risk rate without population/exposure data.

In [79]:
# Figures are separate from CSV export and disabled for this implementation.
if not analysis_ready or not category_ready or not category_inputs_unchanged():
    print("Figures skipped: complete analysis and approved, unchanged category inputs are required.")
elif not create_category_figures:
    print("Figure creation is disabled; no category-dependent plots generated.")
else:
    import matplotlib.pyplot as plt
    period = (eligible_categories["date_posted"].min().strftime("%Y-%m-%d") + " to " +
              eligible_categories["date_posted"].max().strftime("%Y-%m-%d"))
    print("Publication-date range:", period, "| included reports: 5,124")

Figures skipped: complete analysis and approved, unchanged category inputs are required.


In [84]:
if analysis_ready and category_ready and category_inputs_unchanged() and create_category_figures:
    for number, (level, result) in enumerate(analysis_results.items(), start=1):
        totals = result["counts"].groupby("district_clean")["report_count"].sum().sort_values()
        figure_path = figure_folder / ("part_" + str(number) + "_area_report_counts.png")
        if save_category_figures:
            require(not figure_path.exists(), "Existing figure will not be overwritten: " + figure_path.name)
        fig, ax = plt.subplots(figsize=(10, max(5, 0.25 * len(totals))))
        totals.plot.barh(ax=ax, color="#287D8E")
        ax.set(title=level + " — included report counts", xlabel="Reports (not incidents or risk rates)",
               ylabel="Geographic area; Unknown shown separately")
        fig.text(0.01, 0.005, "Publication dates: " + period + " | N = 5,124 | Unknown retained", fontsize=9)
        fig.tight_layout(rect=(0, 0.025, 1, 1))
        if save_category_figures:
            fig.savefig(figure_path, dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)

In [83]:
if analysis_ready and category_ready and category_inputs_unchanged() and create_category_figures:
    for number, (level, result) in enumerate(analysis_results.items(), start=1):
        heatmap = result["counts"].pivot(
            index="district_clean", columns="crime_category", values="share_within_district"
        ).fillna(0)
        figure_path = figure_folder / ("part_" + str(number) + "_category_composition.png")
        if save_category_figures:
            require(not figure_path.exists(), "Existing figure will not be overwritten: " + figure_path.name)
        fig, ax = plt.subplots(figsize=(max(10, 0.8 * len(heatmap.columns)), max(5, 0.25 * len(heatmap))))
        image = ax.imshow(heatmap.to_numpy() * 100, aspect="auto", vmin=0, vmax=100, cmap="Blues")
        ax.set_xticks(range(len(heatmap.columns)), heatmap.columns, rotation=60, ha="right")
        ax.set_yticks(range(len(heatmap.index)), heatmap.index)
        ax.set(title=level + " — category composition within each area", xlabel="Approved crime category", ylabel="Area")
        fig.colorbar(image, ax=ax, label="% of included reports within the area")
        fig.text(0.01, 0.005, "Publication dates: " + period + " | Unknown geography/categories retained", fontsize=9)
        fig.tight_layout(rect=(0, 0.025, 1, 1))
        if save_category_figures:
            fig.savefig(figure_path, dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)

**Visualization and interpretation rules**

After approval validation, the figure switch can create a count chart and category-composition heatmap for each geographic level. Saving PNGs requires a separate save switch; existing filenames are protected. Unknown geography stays visible, and supplied Unknown/Unclear categories stay in the heatmap. All charts identify publication dates, report units and denominators.

Counts describe police reports, not distinct crime incidents, victim counts or population-adjusted risk. date_posted is the publication date; boundary periods are partial. Higher counts may reflect activity, reporting or population differences. No substantive category findings can be written until approved outputs are available.

No true geographic map is implemented without compatible verified official boundaries, source/license information and unmatched-area checks. Stadtteile plus standalone Ortsteile must be labelled exactly; the standalone areas are not assigned invented parent districts. Map preparation remains a separate dependency before final submission.

Current state: Member 06 inputs are absent, so no category-dependent tables, CSVs or figures have been produced. To run later, rerun Section 5 completely, inspect validation, then Sections 6–7. Explicitly enable the CSV/figure switches only in a later authorized run. The six preparation artifacts are never written by these sections.

**What to be careful about:** A district with many reports may have more reporting, activity or residents. Explain observed counts; do not recommend patrol allocations as if this dataset measures complete crime incidence.

## What you must submit to the main workstation

Submit this completed notebook with saved outputs, your files in outputs/member_01, and figures in figures/member_01. Run the notebook top-to-bottom before submission. Do not submit empty templates as results.

- **data_quality.csv** = columns: `check`, `value`, `notes`.
- **district_lookup.csv** = columns: `district_original`, `district_clean`, `decision`, `notes`.
- **duplicate_review.csv** = columns: `report_id`, `url`, `decision`, `reason`.
- **district_category_counts.csv** = columns: `district_clean`, `crime_category`, `report_count`, `share_within_district`, `share_of_category_reports`.
- **highest_district_by_category.csv** = columns: `crime_category`, `district_clean`, `report_count`.
- **run_metadata.json** = name, input checksums, package versions, method settings, run date, record counts and status.
- **handover.md** = what you did, results, limitations, reviewer comments, output inventory and outstanding questions.
- **reports_prepared.parquet** = all base IDs/original columns, with datetime date_posted, district_clean, include_in_analysis and exclusion_reason.

**Main notebook destination:** Data preparation; District distribution and citywide composition.

Submit preparation files first so the group can work in parallel. Submit the district tables/maps after approved categories are available. Keep ties in the highest-district table and show Unknown separately.
### Output conventions

- Keep report_id as the original BR string. Topic IDs are integers within a model, or empty for missing/excluded inputs; HDBSCAN noise is -1.
- For model tables, use the exact model_id and assignment_status values: assigned, unassigned, excluded, missing_text, invalid_embedding. Keywords can be semicolon-separated text. n_reports is the number of reports represented by that topic/status, not a score.
- Store shares/agreement rates as fractions from 0 to 1; multiply by 100 only when displaying percentages. Record which report set is the denominator. A model weight is not a calibrated category probability.
- Date keys use YYYY-MM-DD; monthly keys use YYYY-MM. Keep literal text labels intact when reading lookup CSVs (keep_default_na=False can preserve the original text None).
- Copy templates/member_XX/run_metadata.json into your output folder only when filling it with an actual run. Keep the common metadata keys, replace placeholders, and mark submitted/reviewed only when true. Record SHA-256 file versions, not just filenames.
- Final category statuses are mapped_topic/manual_override/unclear/unassigned/excluded/missing_text/invalid_embedding. Elderly review statuses are yes/no_evidence/unclear/not_reviewed/excluded. Unclear and unreviewed are not negative results.

### Final submission checks

- [ ] All expected output files are populated, or a genuine empty result is explained (for example, no anomalies met the rule).
- [ ] Every report-level table uses report_id and includes the required cohort, without unexpected duplicates.
- [ ] IDs, source/prepared checksums, categories and denominators match the shared contract.
- [ ] Required charts have descriptive titles, labels, units, periods and readable text; export PNG/HTML as appropriate.
- [ ] Code runs in order; results and explanations agree; raw/debug output is bounded.
- [ ] A peer reviewer has checked the method, sample reports, files and conclusions.
- [ ] handover.md says exactly what to import into the main notebook and lists anything still unresolved.

### Your handover summary

**What I completed:** ______  
**Most useful finding (after execution):** ______  
**Output files and row counts:** ______  
**Main limitations:** ______  
**Reviewer decision:** ______  
**Next person who needs my outputs:** ______
